# 📊 Multi-Asset Sector-Constrained Portfolio Optimization

## เป้าหมาย
- ดาวน์โหลดข้อมูลสินทรัพย์ 30 ตัว จากหลากหลาย Asset Class & ภูมิภาค
- รวมหุ้น US, หุ้นไทย, หุ้นจีน, พันธบัตร (US/จีน/ไทย-EM), ทองคำ, Silver
- จำกัดน้ำหนัก (Weight) ในแต่ละกลุ่ม (Sector/Asset Class) เพื่อป้องกันการกระจุกตัว
- เปรียบเทียบพอร์ตที่มี Sector Constraint vs ไม่มี


## 1. Install & Import Libraries


**Environment setup**  
Dependencies live in `pyproject.toml` / `uv.lock`. Run `uv sync` (or install requirements once) before executing instead of using inline `pip install`.


In [ ]:
import json
from pathlib import Path
from datetime import datetime, timedelta

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import yfinance as yf
from pyswarm import pso
from scipy.optimize import minimize
import time
import warnings

warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (14, 7)
plt.rcParams['font.size'] = 12




In [209]:
CONFIG_DIR = Path('config')
SETTINGS_PATH = CONFIG_DIR / 'settings.json'
ASSETS_PATH = CONFIG_DIR / 'assets.csv'

with SETTINGS_PATH.open() as f:
    settings = json.load(f)

BASE_CURRENCY = settings.get('base_currency', 'THB')
RANDOM_SEED = settings.get('random_seed', 42)
np.random.seed(RANDOM_SEED)
rng = np.random.default_rng(RANDOM_SEED)

DATA_CFG = settings.get('data', {})
start_date = datetime.fromisoformat(DATA_CFG['start_date']) if DATA_CFG.get('start_date') else datetime.now() - timedelta(days=365 * 4)
end_date = datetime.fromisoformat(DATA_CFG['end_date']) if DATA_CFG.get('end_date') else datetime.now()

portfolio_cfg = settings.get('portfolio', {})
PORTFOLIO_VALUE = portfolio_cfg.get('value', 1_000_000)

currency_cfg = settings.get('currency', {})
FX_TICKERS = currency_cfg.get('fx_pairs', {})

quality_cfg = settings.get('quality_checks', {})
MAX_ABS_DAILY_RETURN = quality_cfg.get('max_abs_daily_return', 0.25)
MAX_ANNUAL_RETURN = quality_cfg.get('max_annual_return', 1.5)

mc_cfg = settings.get('monte_carlo', {})
NUM_MONTE_CARLO = mc_cfg.get('num_portfolios', 5000)

settings


{'base_currency': 'THB',
 'random_seed': 42,
 'data': {'start_date': '2022-03-21', 'end_date': '2026-03-20'},
 'risk_free_rate': {'ticker': '^TNX', 'fallback': 0.045},
 'portfolio': {'value': 1000000},
 'currency': {'fx_pairs': {'USD': 'USDTHB=X'}},
 'quality_checks': {'max_abs_daily_return': 0.25, 'max_annual_return': 1.5},
 'monte_carlo': {'num_portfolios': 5000}}


## 2. กำหนดสินทรัพย์ 30 ตัว (Multi-Asset, Multi-Region)

ครอบคลุม 9 กลุ่ม: US Tech, Healthcare, Financial, Energy, Consumer, Thai, Chinese, Crypto, Bonds, Commodities


In [210]:
# โหลด Asset Universe จากไฟล์ config
asset_df = pd.read_csv(ASSETS_PATH)
asset_df['Ticker'] = asset_df['Ticker'].str.strip()
asset_df['Currency'] = asset_df['Currency'].str.upper()
asset_df['LotSize'] = asset_df['LotSize'].astype(float)

stock_list = asset_df['Ticker'].tolist()
tickers = stock_list.copy()

stock_sector_map = dict(zip(asset_df['Ticker'], asset_df['Sector']))
currency_map = dict(zip(asset_df['Ticker'], asset_df['Currency']))
lot_size_map = dict(zip(asset_df['Ticker'], asset_df['LotSize']))

print(f"จำนวน Asset ทั้งหมด: {len(stock_list)} ตัว")
sectors = sorted(asset_df['Sector'].unique())
print(f"จำนวนกลุ่ม: {len(sectors)} กลุ่ม")
print()
for s in sectors:
    members = asset_df[asset_df['Sector'] == s]['Ticker'].tolist()
    print(f"  {s:20s} ({len(members)} ตัว): {', '.join(members)}")


จำนวน Asset ทั้งหมด: 30 ตัว
จำนวนกลุ่ม: 9 กลุ่ม
  Bonds                (1 ตัว): TIPS
  Communication Services (1 ตัว): TKO
  Consumer Discretionary (1 ตัว): TJX
  Financials           (2 ตัว): CBOE, IBKR
  Health Care          (4 ตัว): CAH, MCK, GILD, COR
  Industrials          (5 ตัว): FIX, HWM, EME, GE, PWR
  Information Technology (8 ตัว): WDC, PLTR, CIEN, STX, SMCI, GLW, APP, MU
  Thai Stocks          (5 ตัว): KTB.BK, TISCO.BK, TTB.BK, TCAP.BK, SCB.BK
  Utilities            (3 ตัว): VST, CEG, NRG


## 3. ดาวน์โหลดข้อมูลราคา (4 ปี)


In [211]:
print(f"ช่วงเวลา: {start_date.strftime('%Y-%m-%d')} ถึง {end_date.strftime('%Y-%m-%d')}")
print(f"กำลังดาวน์โหลดข้อมูล {len(tickers)} ตัว...\n")

data = yf.download(tickers, start=start_date, end=end_date, auto_adjust=True)['Close']
data = data.dropna()

print(f"\nได้ข้อมูล {data.shape[0]} Daysทำการ, {data.shape[1]} Asset")
print(f"\nAssetที่ได้: {list(data.columns)}")
data.tail()


[*************         27%                       ]  8 of 30 completed


ช่วงเวลา: 2022-03-21 ถึง 2026-03-20
กำลังดาวน์โหลดข้อมูล 30 ตัว...


[*********************100%***********************]  30 of 30 completed


ได้ข้อมูล 910 Daysทำการ, 30 Asset
Assetที่ได้: ['APP', 'CAH', 'CBOE', 'CEG', 'CIEN', 'COR', 'EME', 'FIX', 'GE', 'GILD', 'GLW', 'HWM', 'IBKR', 'KTB.BK', 'MCK', 'MU',
'NRG', 'PLTR', 'PWR', 'SCB.BK', 'SMCI', 'STX', 'TCAP.BK', 'TIPS', 'TISCO.BK', 'TJX', 'TKO', 'TTB.BK', 'VST', 'WDC']


### Currency Conversion (THB Base)
Convert all prices to THB for fair comparison:
- US stocks/ETFs: multiply by USD/THB exchange rate
- Thai stocks (.BK): already in THB
- Crypto: multiply by USD/THB


In [212]:
print(f"Base Currency: {BASE_CURRENCY}")
foreign_currencies = sorted({
    currency_map.get(t, BASE_CURRENCY)
    for t in data.columns
    if currency_map.get(t, BASE_CURRENCY) != BASE_CURRENCY
})
fx_rates = {}

for curr in foreign_currencies:
    fx_ticker = FX_TICKERS.get(curr)
    if not fx_ticker:
        raise ValueError(f"Missing FX ticker mapping for currency {curr} in settings.json")
    print(f"Downloading FX pair {fx_ticker} for {curr}->{BASE_CURRENCY}...")
    fx_df = yf.download(
        fx_ticker,
        start=start_date,
        end=end_date,
        auto_adjust=True,
        progress=False,
    )
    fx_series = fx_df["Close"].squeeze()
    fx_series = fx_series.reindex(data.index).ffill().bfill()
    fx_rates[curr] = fx_series.rename(curr)

convert_count = 0
data_thb = data.copy()

for ticker in data.columns:
    curr = currency_map.get(ticker, BASE_CURRENCY)
    if curr == BASE_CURRENCY:
        continue
    rate_series = fx_rates.get(curr)
    if rate_series is None:
        raise ValueError(f"FX rate for {curr} is missing; please update config.")
    if isinstance(rate_series, pd.DataFrame):
        rate_series = rate_series.squeeze()
    ticker_prices = data[ticker]
    if isinstance(ticker_prices, pd.DataFrame):
        ticker_prices = ticker_prices.squeeze()
    rate_series = rate_series.reindex(data.index).ffill().bfill()
    data_thb[ticker] = ticker_prices * rate_series
    convert_count += 1

print(f"\nConverted {convert_count} assets to {BASE_CURRENCY} base")
print(f"Remaining assets already in {BASE_CURRENCY}: {len(data.columns) - convert_count}")
latest_thb = data_thb.iloc[-1]
print("\nLatest Prices:")
print("-" * 40)
for ticker in sorted(data_thb.columns):
    currency = currency_map.get(ticker, BASE_CURRENCY)
    print(f"  {ticker:12s} {latest_thb[ticker]:>12,.2f} {BASE_CURRENCY}  ({currency})")

data = data_thb
print("\n>> All prices converted to THB base for fair comparison")


Base Currency: THB
Converted 25 assets to THB base
Remaining assets already in THB: 5
Latest Prices:
----------------------------------------
  APP             14,429.38 THB  (USD)
  CAH              6,918.18 THB  (USD)
  CBOE             9,328.65 THB  (USD)
  CEG             10,380.22 THB  (USD)
  CIEN            13,532.62 THB  (USD)
  COR             10,881.07 THB  (USD)
  EME             24,643.62 THB  (USD)
  FIX             47,382.88 THB  (USD)
  GE               9,564.81 THB  (USD)
  GILD             4,628.41 THB  (USD)
  GLW              4,365.02 THB  (USD)
  HWM              7,640.43 THB  (USD)
  IBKR             2,220.23 THB  (USD)
  KTB.BK              34.25 THB  (THB)
  MCK             29,485.56 THB  (USD)
  MU              14,572.06 THB  (USD)
  NRG              5,293.92 THB  (USD)
  PLTR             5,106.30 THB  (USD)
  PWR             18,956.76 THB  (USD)
  SCB.BK             144.00 THB  (THB)
  SMCI             1,009.91 THB  (USD)
  STX             14,254.88 THB  (USD)


## 4. คำนวณ Returns และ Statistics


In [213]:
# Daily returns
returns = data.pct_change().dropna()

# Annualized statistics
annual_returns = returns.mean() * 252
annual_volatility = returns.std() * np.sqrt(252)
sharpe_ratios = annual_returns / annual_volatility

# สร้างตาราง summary
summary = pd.DataFrame({
    'Sector': [stock_sector_map[t] for t in returns.columns], 
    'Annual Return (%)': (annual_returns * 100).round(2), # ผลตอบแทนเฉลี่ยต่อปี
    'Annual Vol (%)': (annual_volatility * 100).round(2), # ความเสี่ยง หรือ ความเหวี่ยงรายปี
    'Sharpe Ratio': sharpe_ratios.round(3) # อัตราส่วนความคุ้มค่าเทียบกับความเสี่ยง -> การหา Annual Return / Annual Vol
})

print("=" * 70)
print("สรุปผลตอบแทนรายปี (Annualized)")
print("=" * 70)
summary.sort_values('Sharpe Ratio', ascending=False)


สรุปผลตอบแทนรายปี (Annualized)


In [214]:
def run_data_quality_checks(price_df, returns_df):
    price_clean = price_df.copy()
    returns_clean = returns_df.copy()
    issues = []
    flagged_assets = set()

    if price_clean.isna().any().any():
        bad_cols = price_clean.columns[price_clean.isna().any()].tolist()
        issues.append(f"พบ NaN ในราคา: {bad_cols}")
        flagged_assets.update(bad_cols)

    daily_extreme = returns_clean.abs().max()
    flagged_daily = daily_extreme[daily_extreme > MAX_ABS_DAILY_RETURN]
    if not flagged_daily.empty:
        issues.append("Daily return เกิน threshold:")
        issues.append(flagged_daily.sort_values(ascending=False))
        flagged_assets.update(flagged_daily.index.tolist())

    annualized = returns_clean.mean() * 252
    flagged_annual = annualized[annualized.abs() > MAX_ANNUAL_RETURN]
    if not flagged_annual.empty:
        issues.append("Annual return เกิน threshold:")
        issues.append(flagged_annual.sort_values(ascending=False))
        flagged_assets.update(flagged_annual.index.tolist())

    if not issues:
        print("✅ ไม่พบปัญหาคุณภาพข้อมูลตาม threshold ที่ตั้งไว้")
    else:
        print("⚠️ Data Quality Alerts:")
        for item in issues:
            print(item)

    if flagged_assets:
        drop_cols = sorted(flagged_assets)
        print(f"🧹 Removing flagged assets from analysis: {drop_cols}")
        price_clean = price_clean.drop(columns=drop_cols, errors='ignore')
        returns_clean = returns_clean.drop(columns=drop_cols, errors='ignore')

    return price_clean, returns_clean

data, returns = run_data_quality_checks(data, returns)


⚠️ Data Quality Alerts:
Daily return เกิน threshold:
Ticker
TIPS      2.968228
APP       0.500886
SCB.BK    0.398773
SMCI      0.357643
NRG       0.327152
PLTR      0.314816
VST       0.289239
FIX       0.263885
dtype: float64
Annual return เกิน threshold:
Ticker
TIPS    6.661081
dtype: float64
🧹 Removing flagged assets from analysis: ['APP', 'FIX', 'NRG', 'PLTR', 'SCB.BK', 'SMCI', 'TIPS', 'VST']


In [215]:
# อัปเดต Asset Universe หลังจากตัดสินทรัพย์ที่ไม่ผ่าน QC
asset_df = asset_df[asset_df['Ticker'].isin(data.columns)].reset_index(drop=True)
stock_list = asset_df['Ticker'].tolist()
tickers = stock_list.copy()
stock_sector_map = dict(zip(asset_df['Ticker'], asset_df['Sector']))
currency_map = dict(zip(asset_df['Ticker'], asset_df['Currency']))
lot_size_map = dict(zip(asset_df['Ticker'], asset_df['LotSize']))

print(f"Asset หลัง QC: {len(stock_list)} ตัว")
for s in sorted(asset_df['Sector'].unique()):
    members = asset_df[asset_df['Sector'] == s]['Ticker'].tolist()
    print(f"  {s:20s}: {members}")


Asset หลัง QC: 22 ตัว
  Communication Services: ['TKO']
  Consumer Discretionary: ['TJX']
  Financials          : ['CBOE', 'IBKR']
  Health Care         : ['CAH', 'MCK', 'GILD', 'COR']
  Industrials         : ['HWM', 'EME', 'GE', 'PWR']
  Information Technology: ['WDC', 'CIEN', 'STX', 'GLW', 'MU']
  Thai Stocks         : ['KTB.BK', 'TISCO.BK', 'TTB.BK', 'TCAP.BK']
  Utilities           : ['CEG']


In [216]:
# Correlation Heatmap แผนภาพความสัมพันธ์ ดูทิศทางของราคาไปในทางเดียวกันหรือสวนทางกัน
fig, ax = plt.subplots(figsize=(16, 12))
corr_matrix = returns.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))  
# สีแดง (ค่าเข้าใกล้ 1): หุ้นสองตัวนี้วิ่งไปในทิศทาง เดียวกัน อย่างรุนแรง
# สีเขียว (ค่าเข้าใกล้ -1): หุ้นสองตัวนี้วิ่ง สวนทางกัน ชัดเจน (ถ้า A ลง B จะขึ้น)
# สีเหลือง (ค่าใกล้ 0): ไม่มีความเกี่ยวข้องกันเลย วิ่งสะเปะสะปะตามใจตัวเอง
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn_r',
            center=0, linewidths=0.5, ax=ax, vmin=-1, vmax=1,
            annot_kws={'size': 7})
ax.set_title('Correlation Matrix: Multi-Asset Portfolio (US, Thai, China, Bonds, Gold, Silver)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


## 5. Mean-Variance (M-V) Model — ทฤษฎี Markowitz (1952)

### ตัวแปรหลัก:
- **Mean (μ)**: ผลตอบแทนที่คาดหวัง — คำนวณจากค่าเฉลี่ยผลตอบแทนในอดีต
- **Variance (σ²)**: ความเสี่ยง — วัดการกระจายตัวของผลตอบแทน

### 2 เป้าหมายของ M-V Model:
1. **Maximize Return** ณ ระดับความเสี่ยงที่ยอมรับได้
2. **Minimize Risk** ณ ระดับผลตอบแทนที่ต้องการ


In [217]:
# =============================================
# Mean-Variance (M-V) Model — Markowitz (1952)
# =============================================
# การลงทุนที่ดีที่สุด ไม่ใช่การเลือกหุ้นที่กำไรมากที่สุด แต่คือการหาสัดส่วนพอร์ตที่ให้ "ผลตอบแทนคุ้มค่าความเสี่ยงที่สุด"

# --- 1) Mean: Expected Return Vector (mu) ---
mu = returns.mean() * 252  # ผลตอบแทนเฉลี่ยต่อปี
print("=" * 60)
print("Mean-Variance Model Components")
print("=" * 60)
print()
print("1) Expected Return Vector (mu) - Annualized:")
print("-" * 40)
for ticker, ret in mu.sort_values(ascending=False).items():
    bar = "#" * int(max(ret * 100, 0) / 3)
    print(f"  {ticker:10s} {ret*100:>8.2f}%  {bar}")

# --- 2) Covariance Matrix (Sigma) ---
cov = returns.cov() * 252  # ความเสี่ยง
print()
print("2) Covariance Matrix (Sigma) - shape:", cov.shape)
print("-" * 40)
print("  Diagonal = Variance of each asset:")
for ticker in mu.sort_values(ascending=False).index:
    var_val = cov.loc[ticker, ticker]
    vol_val = np.sqrt(var_val) * 100
    print(f"    {ticker:10s}  Var={var_val:.4f}  Vol={vol_val:.1f}%")

# --- 3) Portfolio Formulas ---
print()
print("3) M-V Model Formulas:")
print("-" * 40)
print("  Portfolio Return:  E(Rp) = sum(wi * mui)") #  (ผลตอบแทนพอร์ต): เอา (สัดส่วนน้ำหนักหุ้น A * ผลตอบแทนหุ้น A) + (สัดส่วนหุ้น B * ผลตอบแทน B) ... 
print("  Portfolio Risk:    Vp    = w.T @ Sigma @ w") # (ความต้านทานความเสี่ยง ): (น้ำหนักพอร์ตแบบ Transpose แนวนอน) คูณเมทริกซ์ (ตาราง Sigma) คูณ (น้ำหนักแนวดิ่ง)
print("  Sharpe Ratio:      SR    = E(Rp) / sqrt(Vp)") # ผลเฉลี่ยพอร์ต หาร สแควรูทของความเสี่ยง

# --- 4) Equal-Weight Portfolio Example --- จำลองพอร์ตแบบ "น้ำหนักเท่ากันทุกตัว" (Equal-Weight)
n = len(mu)
w_eq = np.array([1/n] * n)
port_ret = np.dot(w_eq, mu)
port_var = np.dot(w_eq.T, np.dot(cov, w_eq))
port_vol = np.sqrt(port_var)
port_sr = port_ret / port_vol

print()
print("4) Example: Equal-Weight Portfolio (w = 1/N):")
print("-" * 40)
print(f"  E(Rp) = sum(wi * mui)      = {port_ret*100:.2f}%")
print(f"  Vp    = w.T @ Sigma @ w    = {port_var:.6f}") # ความแปรปรวน/ความเสี่ยงของพอร์ต
print(f"  Vol   = sqrt(Vp)           = {port_vol*100:.2f}%") # ความผันผวนของพอร์ตคาดการณ์ต่อปี
print(f"  Sharpe = E(Rp) / Vol       = {port_sr:.3f}") # อัตราส่วนผลตอบแทนเทียบกับความเสี่ยง

# --- 5) Optimization with Constraints ---
print()


Mean-Variance Model Components
1) Expected Return Vector (mu) - Annualized:
----------------------------------------
  WDC           71.97%  #######################
  CIEN          66.80%  ######################
  MU            64.38%  #####################
  STX           58.81%  ###################
  EME           58.00%  ###################
  CEG           57.16%  ###################
  HWM           56.43%  ##################
  GE            50.47%  ################
  PWR           46.20%  ###############
  IBKR          45.77%  ###############
  GLW           45.69%  ###############
  TKO           40.16%  #############
  CAH           38.12%  ############
  KTB.BK        34.54%  ###########
  MCK           31.25%  ##########
  GILD          29.21%  #########
  CBOE          27.62%  #########
  TJX           26.80%  ########
  TTB.BK        23.67%  #######
  COR           22.35%  #######
  TCAP.BK       16.65%  #####
  TISCO.BK      13.20%  ####
2) Covariance Matrix (Sigma) - shape

In [218]:
# ==========================================================
# หัวใจ M-V Model: Correlation/Covariance ระหว่างAsset
# ==========================================================
# ไม่ใช่แค่เลือกหุ้นกำไรดี แต่คือการดู "ความสัมพันธ์" ระหว่างAsset
# - หากลงหุ้นที่ขึ้นลงพร้อมกัน -> พอร์ตเสี่ยงสูง
# - หากเลือกAssetที่ "ไม่ไปด้วยกัน" -> พอร์ตเสี่ยงน้อยลง ผลตอบแทนเท่าเดิม
# ==========================================================

mu = returns.mean() * 252
cov = returns.cov() * 252
corr = returns.corr()

# --- 1) แสดงตัวอย่าง Correlation สูง vs ต่ำ ---
print("=" * 65)
print("หัวใจ M-V Model: ทำไม Correlation สำคัญ?")
print("=" * 65)

# หา pairs ที่ Corr สูงสุด และ ต่ำสุด -> ค้นหาคู่หุ้นที่ Correlation ต่ำสุด และ สูงสุด
corr_pairs = []
tickers_list = list(corr.columns)
for i in range(len(tickers_list)):
    for j in range(i+1, len(tickers_list)):
        corr_pairs.append((tickers_list[i], tickers_list[j], corr.iloc[i,j]))

corr_pairs.sort(key=lambda x: x[2])

print()
print("Correlation ต่ำสุด (diversify ได้ดี):")
for a, b, c in corr_pairs[:5]:
    print(f"  {a:10s} vs {b:10s}  corr = {c:+.3f}  {"<-- negative!" if c < 0 else ""}")

print()
print("Correlation สูงสุด (เสี่ยงกระจุก):")
for a, b, c in corr_pairs[-5:]:
    print(f"  {a:10s} vs {b:10s}  corr = {c:+.3f}")

# --- 2) พิสูจน์ด้วยตัวเลข: 2-asset portfolio ---
print()
print("=" * 65)
print("พิสูจน์: เปรียบเทียบ 2-asset portfolio")
print("=" * 65)

# หา high-corr pair & low-corr pair
high_a, high_b, high_c = corr_pairs[-1]
low_a, low_b, low_c = corr_pairs[0]

def two_asset_portfolio(t1, t2, w1=0.5):
    w2 = 1 - w1
    ret = w1 * mu[t1] + w2 * mu[t2]
    var = (w1**2 * cov.loc[t1,t1] + w2**2 * cov.loc[t2,t2]
           + 2 * w1 * w2 * cov.loc[t1,t2])
    vol = np.sqrt(var)
    avg_vol = w1 * np.sqrt(cov.loc[t1,t1]) + w2 * np.sqrt(cov.loc[t2,t2])
    diversification = (1 - vol/avg_vol) * 100
    return ret, vol, avg_vol, diversification

ret_h, vol_h, avg_vol_h, div_h = two_asset_portfolio(high_a, high_b)
ret_l, vol_l, avg_vol_l, div_l = two_asset_portfolio(low_a, low_b)

print(f"\nCase 1: HIGH Corr ({high_a} + {high_b}, corr={high_c:+.3f})")
print(f"  Average Vol (ไม่มี diversification) = {avg_vol_h*100:.2f}%")
print(f"  Portfolio Vol (50/50)               = {vol_h*100:.2f}%")
print(f"  Diversification Benefit             = {div_h:.1f}%  <-- น้อย!")

print(f"\nCase 2: LOW Corr ({low_a} + {low_b}, corr={low_c:+.3f})")
print(f"  Average Vol (ไม่มี diversification) = {avg_vol_l*100:.2f}%")
print(f"  Portfolio Vol (50/50)               = {vol_l*100:.2f}%")
print(f"  Diversification Benefit             = {div_l:.1f}%  <-- สูงกว่ามาก!")

print(f"\n>> สรุป: Corr ต่ำ -> ลดความเสี่ยงได้ {div_l:.1f}% vs Corr สูง -> ลดได้แค่ {div_h:.1f}%")

# --- 3) Visualization ---
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Heatmap: Asset Class avg correlation
sector_list = sorted(set(stock_sector_map.values()))
sector_corr = pd.DataFrame(index=sector_list, columns=sector_list, dtype=float)
for s1 in sector_list:
    t1s = [t for t in corr.columns if stock_sector_map.get(t) == s1]
    for s2 in sector_list:
        t2s = [t for t in corr.columns if stock_sector_map.get(t) == s2]
        vals = [corr.loc[a,b] for a in t1s for b in t2s if a != b]
        sector_corr.loc[s1, s2] = np.mean(vals) if vals else (1.0 if s1==s2 else 0.0)

sns.heatmap(sector_corr.astype(float), annot=True, fmt=".2f", cmap="RdYlGn_r",
            center=0, vmin=-0.5, vmax=1, ax=axes[0], linewidths=1)
axes[0].set_title("Avg Correlation Between Asset Groups\n(Lower = Better Diversification)", fontsize=12, fontweight="bold")

# Bar chart: diversification benefit
weights_range = np.linspace(0, 1, 50)
vols_high = [two_asset_portfolio(high_a, high_b, w)[1]*100 for w in weights_range]
vols_low = [two_asset_portfolio(low_a, low_b, w)[1]*100 for w in weights_range]

axes[1].plot(weights_range*100, vols_high, "r-", linewidth=2,
             label=f"High Corr: {high_a}+{high_b} ({high_c:+.2f})")
axes[1].plot(weights_range*100, vols_low, "g-", linewidth=2,
             label=f"Low Corr: {low_a}+{low_b} ({low_c:+.2f})")
axes[1].set_xlabel("Weight of Asset 1 (%)", fontsize=12)  # น้ำหนักของหุ้นตัว
axes[1].set_ylabel("Portfolio Volatility (%)", fontsize=12) # ความเสี่ยงของพอร์ตตัวรวม
axes[1].set_title("Diversification Effect\n(Low Corr = More Risk Reduction)", fontsize=12, fontweight="bold")
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


หัวใจ M-V Model: ทำไม Correlation สำคัญ?
Correlation ต่ำสุด (diversify ได้ดี):
  GE         vs KTB.BK      corr = -0.095  <-- negative!
  EME        vs KTB.BK      corr = -0.087  <-- negative!
  KTB.BK     vs PWR         corr = -0.073  <-- negative!
  GILD       vs TCAP.BK     corr = -0.055  <-- negative!
  COR        vs KTB.BK      corr = -0.048  <-- negative!
Correlation สูงสุด (เสี่ยงกระจุก):
  CAH        vs COR         corr = +0.678
  EME        vs PWR         corr = +0.709
  MU         vs WDC         corr = +0.712
  COR        vs MCK         corr = +0.752
  STX        vs WDC         corr = +0.802
พิสูจน์: เปรียบเทียบ 2-asset portfolio
Case 1: HIGH Corr (STX + WDC, corr=+0.802)
  Average Vol (ไม่มี diversification) = 49.70%
  Portfolio Vol (50/50)               = 47.19%
  Diversification Benefit             = 5.1%  <-- น้อย!
Case 2: LOW Corr (GE + KTB.BK, corr=-0.095)
  Average Vol (ไม่มี diversification) = 28.52%
  Portfolio Vol (50/50)               = 19.45%
  Diversification Ben

## 6. กำหนด Sector Weight Limits (ข้อจำกัดน้ำหนัก)

### กฎ:
- **น้ำหนักสูงสุดต่อกลุ่ม**: ไม่เกิน 25%
- **น้ำหนักต่ำสุดต่อกลุ่ม**: อย่างน้อย 2%
- **น้ำหนักสูงสุดต่อตัว**: ไม่เกิน 15%
- **น้ำหนักต่ำสุดต่อตัว**: อย่างน้อย 1%


In [219]:
# === Sector Weight Constraints Configuration ===

# น้ำหนักรวมของแต่ละกลุ่ม
SECTOR_MAX_WEIGHT = 0.25   # สูงสุด 25% ต่อกลุ่ม
SECTOR_MIN_WEIGHT = 0.02   # ต่ำสุด 2% ต่อกลุ่ม

# น้ำหนักของAssetแต่ละตัว
STOCK_MAX_WEIGHT = 0.15    # สูงสุด 15% ต่อตัว
STOCK_MIN_WEIGHT = 0.01    # ต่ำสุด 1% ต่อตัว

print("=" * 55)
print("📋 Sector / Asset Class Weight Constraints")
print("=" * 55)
print(f"  Sector Max Weight : {SECTOR_MAX_WEIGHT*100:.0f}%")
print(f"  Sector Min Weight : {SECTOR_MIN_WEIGHT*100:.0f}%")
print(f"  Stock Max Weight  : {STOCK_MAX_WEIGHT*100:.0f}%")
print(f"  Stock Min Weight  : {STOCK_MIN_WEIGHT*100:.0f}%")
print()

# สร้าง Sector mapping
unique_sectors = sorted(set(stock_sector_map.values()))
stock_list = list(returns.columns)
n_stocks = len(stock_list)

# สร้าง sector index mapping
sector_stock_indices = {}
for sector in unique_sectors:
    indices = [i for i, s in enumerate(stock_list) if stock_sector_map[s] == sector]
    sector_stock_indices[sector] = indices
    print(f"  {sector:20s}: {[stock_list[i] for i in indices]}")


📋 Sector / Asset Class Weight Constraints
  Sector Max Weight : 25%
  Sector Min Weight : 2%
  Stock Max Weight  : 15%
  Stock Min Weight  : 1%
  Communication Services: ['TKO']
  Consumer Discretionary: ['TJX']
  Financials          : ['CBOE', 'IBKR']
  Health Care         : ['CAH', 'COR', 'GILD', 'MCK']
  Industrials         : ['EME', 'GE', 'HWM', 'PWR']
  Information Technology: ['CIEN', 'GLW', 'MU', 'STX', 'WDC']
  Thai Stocks         : ['KTB.BK', 'TCAP.BK', 'TISCO.BK', 'TTB.BK']
  Utilities           : ['CEG']


In [220]:
# ==========================================================
# Risk-Free Rate (Rf) for Sharpe Ratio
# ==========================================================
# Sharpe Ratio = (E(Rp) - Rf) / sigma_p
# Using US 10-Year Treasury Rate as proxy

import yfinance as yf

try:
    # Download US 10Y Treasury yield
    tnx = yf.download("^TNX", period="5d", auto_adjust=True)["Close"]
    RISK_FREE_RATE = float(tnx.iloc[-1]) / 100  # convert from % to decimal
except:
    RISK_FREE_RATE = 0.045  # fallback: 4.5%

print("=" * 60)
print("Risk-Free Rate Configuration")
print("=" * 60)
print(f"  Rf (annual) = {RISK_FREE_RATE*100:.2f}%  (US 10Y Treasury)")  # ดอกเบี้ยแบบรายปี -> พันธบัตรรัฐบาลสหรัฐอเมริกาอายุ 10 ปี
print(f"  Rf (daily)  = {RISK_FREE_RATE/252*100:.4f}%") # ดอกเบี้ยผลตอบแทนตกแบบรายวัน
print()
print("  Sharpe Ratio formula:")
print("    SR = (E(Rp) - Rf) / sigma_p")
print(f"    (previously used Rf=0, now Rf={RISK_FREE_RATE*100:.2f}%)")


[*********************100%***********************]  1 of 1 completed


Risk-Free Rate Configuration
  Rf (annual) = 4.50%  (US 10Y Treasury)
  Rf (daily)  = 0.0179%
  Sharpe Ratio formula:
    SR = (E(Rp) - Rf) / sigma_p
    (previously used Rf=0, now Rf=4.50%)


## 7. Portfolio Optimization Functions


In [221]:
# === Optimization Helper Functions ===

def portfolio_performance(weights, mean_returns, cov_matrix):
    """คำนวณผลตอบแทนและความเสี่ยงของพอร์ต"""
    port_return = np.sum(mean_returns * weights) * 252
    port_volatility = np.sqrt(np.dot(weights.T, np.dot(cov_matrix * 252, weights)))
    sharpe = (port_return - RISK_FREE_RATE) / port_volatility
    return port_return, port_volatility, sharpe


def neg_sharpe_ratio(weights, mean_returns, cov_matrix):
    """Objective function: minimize negative Sharpe Ratio"""
    ret, vol, sharpe = portfolio_performance(weights, mean_returns, cov_matrix)
    return -sharpe


def minimize_volatility(weights, mean_returns, cov_matrix):
    """Objective function: minimize volatility"""
    return portfolio_performance(weights, mean_returns, cov_matrix)[1]


def get_base_constraints():
    """Constraints ทั่วไป: น้ำหนักรวม = 100%"""
    return [{'type': 'eq', 'fun': lambda w: np.sum(w) - 1}]


def get_sector_constraints(sector_stock_indices, sector_max, sector_min):
    """สร้าง Constraints จำกัดน้ำหนักรายกลุ่ม (Sector/Asset Class)"""
    constraints = []
    for sector, indices in sector_stock_indices.items():
        constraints.append({'type': 'ineq', 'fun': lambda w, idx=indices: sector_max - np.sum(w[idx])})
        constraints.append({'type': 'ineq', 'fun': lambda w, idx=indices: np.sum(w[idx]) - sector_min})
    return constraints


def optimize_portfolio(mean_returns, cov_matrix, objective_func,
                       sector_constraint=False, n_trials=50):
    """Optimize portfolio with or without sector constraints"""
    n = len(mean_returns)
    if sector_constraint:
        bounds = tuple((STOCK_MIN_WEIGHT, STOCK_MAX_WEIGHT) for _ in range(n))
    else:
        bounds = tuple((0, 1) for _ in range(n))

    constraints = get_base_constraints()
    if sector_constraint:
        constraints += get_sector_constraints(
            sector_stock_indices, SECTOR_MAX_WEIGHT, SECTOR_MIN_WEIGHT
        )

    best_result = None
    best_obj = np.inf

    for _ in range(n_trials):
        init_weights = rng.dirichlet(np.ones(n))
        try:
            result = minimize(
                objective_func,
                init_weights,
                args=(mean_returns, cov_matrix),
                method='SLSQP',
                bounds=bounds,
                constraints=constraints,
                options={'maxiter': 1000, 'ftol': 1e-12}
            )
        except Exception:
            continue
        fun_val = getattr(result, 'fun', np.inf)
        if np.isfinite(fun_val) and fun_val < best_obj:
            best_obj = fun_val
            best_result = result

    if best_result is None:
        raise RuntimeError("Portfolio optimization failed. Please relax constraints or adjust inputs.")

    lower = np.array([b[0] for b in bounds])
    upper = np.array([b[1] for b in bounds])
    cleaned = np.clip(best_result.x, lower, upper)
    total = cleaned.sum()
    if total <= 0:
        cleaned = np.ones_like(cleaned) / len(cleaned)
    else:
        cleaned = cleaned / total
    best_result.x = cleaned

    if not getattr(best_result, 'success', False):
        status = getattr(best_result, 'message', 'Unknown status')
        print(f"⚠️ SLSQP returned without success ({status}). Using best feasible weights found.")

    return best_result

print("✅ Functions defined successfully")


✅ Functions defined successfully


## 8. Optimize: Maximum Sharpe Ratio Portfolio


In [222]:
mean_returns = returns.mean()
cov_matrix = returns.cov()

# === 1) ไม่มี Sector Constraint (Unconstrained) ===
print("⏳ Optimizing: Max Sharpe (Unconstrained)...")
result_unconstrained = optimize_portfolio(
    mean_returns, cov_matrix, neg_sharpe_ratio, sector_constraint=False
)
w_unconstrained = result_unconstrained.x
ret_uc, vol_uc, sharpe_uc = portfolio_performance(w_unconstrained, mean_returns, cov_matrix)
print(f"  Return: {ret_uc*100:.2f}% | Vol: {vol_uc*100:.2f}% | Sharpe: {sharpe_uc:.3f}")

# === 2) มี Sector Constraint ===
print("\n⏳ Optimizing: Max Sharpe (Sector Constrained)...")
t0_slsqp = time.time()
result_constrained = optimize_portfolio(
    mean_returns, cov_matrix, neg_sharpe_ratio, sector_constraint=True
)
w_constrained = result_constrained.x
ret_c, vol_c, sharpe_c = portfolio_performance(w_constrained, mean_returns, cov_matrix)
print(f"  Return: {ret_c*100:.2f}% | Vol: {vol_c*100:.2f}% | Sharpe: {sharpe_c:.3f}")

print("\n✅ Optimization complete!")


⏳ Optimizing: Max Sharpe (Unconstrained)...
  Return: 34.80% | Vol: 12.01% | Sharpe: 2.523
⏳ Optimizing: Max Sharpe (Sector Constrained)...
  Return: 40.04% | Vol: 15.05% | Sharpe: 2.361
✅ Optimization complete!


### PSO (Particle Swarm Optimization)

PSO is a **metaheuristic** optimization algorithm inspired by bird flocking behavior:
- **Particles** = candidate portfolios (each particle = set of weights)
- **Swarm** = population of particles exploring the solution space
- Each particle updates its position based on:
  1. **Personal best** (pbest) — best solution this particle has found
  2. **Global best** (gbest) — best solution any particle has found

**Advantages over SLSQP:**
- Does not require gradient computation
- Better at escaping local optima
- Can handle non-convex objective functions

**Disadvantage:**
- Slower (more function evaluations)
- Cannot enforce equality constraints directly (uses penalty method)


In [223]:
# ==========================================================
# PSO (Particle Swarm Optimization) — Improved Version
# ==========================================================
from pyswarm import pso
from scipy.optimize import minimize as sp_minimize

mu_annual = returns.mean() * 252
cov_annual = returns.cov() * 252
n_assets = len(mu_annual)

def pso_objective(w):
    """Negative Sharpe Ratio with penalty for constraint violations"""
    w_norm = w / np.sum(w) if np.sum(w) > 0 else np.ones(n_assets) / n_assets

    port_ret = np.dot(w_norm, mu_annual)
    port_vol = np.sqrt(np.dot(w_norm.T, np.dot(cov_annual, w_norm)))

    if port_vol < 1e-10:
        return 999

    sharpe = (port_ret - RISK_FREE_RATE) / port_vol

    penalty = 0
    for sector in set(stock_sector_map.values()):
        idx = [i for i, t in enumerate(stock_list) if stock_sector_map[t] == sector]
        sector_w = np.sum(w_norm[idx])
        if sector_w > SECTOR_MAX_WEIGHT:
            penalty += 100 * (sector_w - SECTOR_MAX_WEIGHT)**2
        if sector_w < SECTOR_MIN_WEIGHT:
            penalty += 100 * (SECTOR_MIN_WEIGHT - sector_w)**2

    for wi in w_norm:
        if wi > STOCK_MAX_WEIGHT:
            penalty += 50 * (wi - STOCK_MAX_WEIGHT)**2
        if wi < STOCK_MIN_WEIGHT:
            penalty += 50 * (STOCK_MIN_WEIGHT - wi)**2

    return -sharpe + penalty

lb = np.ones(n_assets) * STOCK_MIN_WEIGHT
ub = np.ones(n_assets) * STOCK_MAX_WEIGHT

print("=" * 70)
print("PSO (Improved) — Particle Swarm Optimization")
print("=" * 70)
print("  Config: 200 particles x 500 iterations x 3 restarts")
print(f"  Assets: {n_assets} | Rf: {RISK_FREE_RATE*100:.2f}%")
print()

start_time = time.time()
best_sharpe = -np.inf
best_w = None

for run in range(3):
    w_raw, fopt = pso(pso_objective, lb, ub,
                      swarmsize=200, maxiter=500,
                      minstep=1e-12, minfunc=1e-12,
                      debug=False)
    w_pso_norm = w_raw / np.sum(w_raw)
    pso_ret_run = np.dot(w_pso_norm, mu_annual)
    pso_vol_run = np.sqrt(np.dot(w_pso_norm.T, np.dot(cov_annual, w_pso_norm)))
    pso_sr_run = (pso_ret_run - RISK_FREE_RATE) / pso_vol_run
    print(f"  Run {run+1}: PSO Sharpe={pso_sr_run:.4f}")
    if pso_sr_run > best_sharpe:
        best_sharpe = pso_sr_run
        best_w = w_pso_norm

if best_w is None:
    raise RuntimeError("PSO failed to produce a feasible solution. Please review constraints.")

pso_time = time.time() - start_time
w_pso = best_w
pso_ret = np.dot(w_pso, mu_annual)
pso_vol = np.sqrt(np.dot(w_pso.T, np.dot(cov_annual, w_pso)))
pso_sharpe = (pso_ret - RISK_FREE_RATE) / pso_vol

slsqp_ret = np.dot(w_constrained, mu_annual)
slsqp_vol = np.sqrt(np.dot(w_constrained.T, np.dot(cov_annual, w_constrained)))
slsqp_sharpe = (slsqp_ret - RISK_FREE_RATE) / slsqp_vol

print(f"\n  Best PSO Sharpe: {best_sharpe:.4f} (in {pso_time:.1f} sec)")
print()

print("=" * 70)
print("SLSQP vs PSO: Comparison")
print("=" * 70)
comp_data = {
    "Metric": ["Annual Return (%)", "Annual Volatility (%)", "Sharpe Ratio (Rf-adj)", "Optimization Time"],
    "SLSQP": [f"{slsqp_ret*100:.2f}", f"{slsqp_vol*100:.2f}", f"{slsqp_sharpe:.3f}", "< 1 sec"],
    "PSO": [f"{pso_ret*100:.2f}", f"{pso_vol*100:.2f}", f"{pso_sharpe:.3f}", f"{pso_time:.1f} sec"],
}
comp_df = pd.DataFrame(comp_data)
print(comp_df.to_string(index=False))

if abs(pso_sharpe - slsqp_sharpe) < 1e-6:
    print("\n  >> ทั้งสองวิธีให้ผล Sharpe เท่ากันในทางปฏิบัติ (ต่างกันน้อยกว่า 1e-6)")
elif pso_sharpe > slsqp_sharpe:
    print("\n  >> PSO ดีกว่า SLSQP ในรอบนี้")
else:
    print("\n  >> SLSQP ดีกว่า PSO ในรอบนี้")


PSO (Improved) — Particle Swarm Optimization
  Config: 200 particles x 500 iterations x 3 restarts
  Assets: 22 | Rf: 4.50%
Stopping search: Swarm best objective change less than 1e-12
  Run 1: PSO Sharpe=2.3137
Stopping search: Swarm best objective change less than 1e-12
  Run 2: PSO Sharpe=2.3026
Stopping search: Swarm best objective change less than 1e-12
  Run 3: PSO Sharpe=2.3269
  Best PSO Sharpe: 2.3269 (in 12.4 sec)
SLSQP vs PSO: Comparison
               Metric   SLSQP      PSO
    Annual Return (%)   40.04    39.33
Annual Volatility (%)   15.05    14.97
Sharpe Ratio (Rf-adj)   2.361    2.327
    Optimization Time < 1 sec 12.4 sec
  >> SLSQP ดีกว่า PSO ในรอบนี้


In [224]:
# ==========================================================
# Differential Evolution (DE) — Pure Evolutionary Search
# ==========================================================
from scipy.optimize import differential_evolution

bounds_de = [(STOCK_MIN_WEIGHT, STOCK_MAX_WEIGHT) for _ in range(n_assets)]

print("=" * 70)
print("DE (Differential Evolution)")
print("=" * 70)
print("  Config: popsize=15 x maxiter=100 x 3 restarts")
print()

start_time = time.time()
best_de_sharpe = -np.inf
best_de_w = None

for run in range(3):
    res_de = differential_evolution(
        pso_objective,
        bounds_de,
        maxiter=100,
        popsize=15,
        mutation=(0.5, 1.0),
        recombination=0.7,
        disp=False,
        polish=False,
    )
    w_de_raw = res_de.x
    w_de_norm = w_de_raw / np.sum(w_de_raw) if np.sum(w_de_raw) > 0 else np.ones(n_assets) / n_assets
    de_ret_run = np.dot(w_de_norm, mu_annual)
    de_vol_run = np.sqrt(np.dot(w_de_norm.T, np.dot(cov_annual, w_de_norm)))
    de_sr_run = (de_ret_run - RISK_FREE_RATE) / de_vol_run
    print(f"  Run {run+1}: DE Sharpe={de_sr_run:.4f}")
    if de_sr_run > best_de_sharpe:
        best_de_sharpe = de_sr_run
        best_de_w = w_de_norm

if best_de_w is None:
    raise RuntimeError("DE failed to produce a feasible solution. Please review constraints.")

de_time = time.time() - start_time
w_de = best_de_w
de_ret = np.dot(w_de, mu_annual)
de_vol = np.sqrt(np.dot(w_de.T, np.dot(cov_annual, w_de)))
de_sharpe = (de_ret - RISK_FREE_RATE) / de_vol

print(f"\n  Best DE Sharpe: {best_de_sharpe:.4f} (in {de_time:.1f} sec)")
print()

print("=" * 80)
print("ULTIMATE COMPARISON: SLSQP vs PSO vs DE")
print("=" * 80)
comp_data = {
    "Metric": ["Annual Return (%)", "Annual Volatility (%)", "Sharpe Ratio (Rf-adj)", "Time"],
    "SLSQP": [f"{slsqp_ret*100:.2f}", f"{slsqp_vol*100:.2f}", f"{slsqp_sharpe:.3f}", "< 1s"],
    "PSO": [f"{pso_ret*100:.2f}", f"{pso_vol*100:.2f}", f"{pso_sharpe:.3f}", f"{pso_time:.1f}s"],
    "DE": [f"{de_ret*100:.2f}", f"{de_vol*100:.2f}", f"{de_sharpe:.3f}", f"{de_time:.1f}s"],
}
comp_df = pd.DataFrame(comp_data)
print(comp_df.to_string(index=False))

scores = {"SLSQP": slsqp_sharpe, "PSO": pso_sharpe, "DE": de_sharpe}
winner = max(scores, key=scores.get)
print(f"\n  🏆 ULTIMATE WINNER: {winner} (Sharpe = {scores[winner]:.4f})")

best_optimizer_label = winner
best_optimizer_weights = {
    "SLSQP": w_constrained,
    "PSO": w_pso,
    "DE": w_de,
}[winner].copy()


DE (Differential Evolution)
  Config: popsize=15 x maxiter=100 x 3 restarts
  Run 1: DE Sharpe=2.2896
  Run 2: DE Sharpe=2.2978
  Run 3: DE Sharpe=2.2721
  Best DE Sharpe: 2.2978 (in 2.8 sec)
ULTIMATE COMPARISON: SLSQP vs PSO vs DE
               Metric SLSQP   PSO    DE
    Annual Return (%) 40.04 39.33 39.67
Annual Volatility (%) 15.05 14.97 15.30
Sharpe Ratio (Rf-adj) 2.361 2.327 2.298
                 Time  < 1s 12.4s  2.8s
  🏆 ULTIMATE WINNER: SLSQP (Sharpe = 2.3609)


In [225]:
# ==================================================================
# PROPAGATE WINNER WEIGHTS FOR DOWNSTREAM ANALYSIS
# ==================================================================
w_constrained_old = w_constrained.copy()  # Backup original SLSQP weights
w_constrained = best_optimizer_weights.copy()
print(f"\n\n[SYSTEM] Assigned w_constrained = {best_optimizer_label} weights for all downstream analysis!")


[SYSTEM] Assigned w_constrained = SLSQP weights for all downstream analysis!


In [226]:
# === Efficient Frontier & Monte Carlo Simulation ===
import matplotlib.cm as cm

print("Generating Efficient Frontier (Monte Carlo 5,000 portfolios)...")

num_portfolios = 5000
mc_results = np.zeros((3, num_portfolios))

for i in range(num_portfolios):
    weights = rng.random(n_assets)
    weights /= np.sum(weights)
    
    p_ret = np.sum(mu_annual * weights)
    p_std = np.sqrt(np.dot(weights.T, np.dot(cov_annual, weights)))
    
    mc_results[0,i] = p_std
    mc_results[1,i] = p_ret
    mc_results[2,i] = (p_ret - RISK_FREE_RATE) / p_std

plt.figure(figsize=(12, 8))
plt.scatter(mc_results[0,:]*100, mc_results[1,:]*100, c=mc_results[2,:], 
            cmap='viridis', marker='o', s=10, alpha=0.3)
plt.colorbar(label='Sharpe Ratio (Simulated)')

# Mark Optimized Portfolios
plt.scatter(slsqp_vol*100, slsqp_ret*100, marker='*', s=300, color='#2196F3', edgecolors='white', label='SLSQP')
plt.scatter(pso_vol*100, pso_ret*100, marker='*', s=300, color='#FF9800', edgecolors='white', label='PSO')
plt.scatter(de_vol*100, de_ret*100, marker='*', s=500, color='#4CAF50', edgecolors='black', label='DE (Best)')

plt.title('Efficient Frontier: Monte Carlo Simulation vs Optimizers', fontsize=14, fontweight='bold')
plt.xlabel('Annual Volatility (%)', fontsize=12)
plt.ylabel('Annual Return (%)', fontsize=12)
plt.legend(fontsize=10, loc='best')
plt.grid(True, alpha=0.2)
plt.show()


Generating Efficient Frontier (Monte Carlo 5,000 portfolios)...


## 9. เปรียบเทียบ Sector/Asset Class Weights


In [227]:
def get_sector_weights(weights, stock_list, stock_sector_map):
    """คำนวณน้ำหนักรวมแต่ละกลุ่ม"""
    sector_w = {}
    for i, stock in enumerate(stock_list):
        sector = stock_sector_map[stock]
        sector_w[sector] = sector_w.get(sector, 0) + weights[i]
    return pd.Series(sector_w)

sector_w_uc = get_sector_weights(w_unconstrained, stock_list, stock_sector_map)
sector_w_slsqp = get_sector_weights(w_constrained_old, stock_list, stock_sector_map)
sector_w_pso = get_sector_weights(w_pso, stock_list, stock_sector_map)
sector_w_de = get_sector_weights(w_de, stock_list, stock_sector_map)

# สร้าง DataFrame เปรียบเทียบ
sector_comp = pd.DataFrame({
    "Unconstrained (%)": (sector_w_uc * 100).round(2),
    "SLSQP (%)": (sector_w_slsqp * 100).round(2),
    "PSO (%)": (sector_w_pso * 100).round(2),
    "DE (%)": (sector_w_de * 100).round(2),
    "Limit (%)": SECTOR_MAX_WEIGHT * 100
}).fillna(0)

print("=" * 80)
print("📊 Sector Weights: Unconstrained vs SLSQP vs PSO vs DE")
print("=" * 80)
print(sector_comp.to_string())


📊 Sector Weights: Unconstrained vs SLSQP vs PSO vs DE
                        Unconstrained (%)  SLSQP (%)  PSO (%)  DE (%)  Limit (%)
Health Care                         11.64      19.61    22.34   18.37       25.0
Financials                          12.41      16.00    12.85   13.08       25.0
Utilities                            0.00       2.00     1.66    1.25       25.0
Information Technology               6.28       9.56     9.38   13.74       25.0
Industrials                         17.96      20.66    20.88   20.57       25.0
Thai Stocks                         48.55      25.00    25.61   25.91       25.0
Consumer Discretionary               0.00       2.00     1.98    3.77       25.0
Communication Services               3.15       5.17     5.31    3.30       25.0


In [228]:
# === Visualization: Sector Weights Comparison ===

fig, ax = plt.subplots(figsize=(16, 8))

sectors = sector_comp.index
x = np.arange(len(sectors))
bw = 0.2  # bar width

ax.bar(x - 1.5*bw, sector_comp["Unconstrained (%)"], bw, label="Unconstrained", color="#9E9E9E", alpha=0.7)
ax.bar(x - 0.5*bw, sector_comp["SLSQP (%)"], bw, label="SLSQP", color="#2196F3", alpha=0.9)
ax.bar(x + 0.5*bw, sector_comp["PSO (%)"], bw, label="PSO", color="#FF9800", alpha=0.9)
ax.bar(x + 1.5*bw, sector_comp["DE (%)"], bw, label="DE (Winner)", color="#4CAF50", alpha=0.9)

# เส้น Limit
ax.axhline(y=SECTOR_MAX_WEIGHT*100, color="red", linestyle="--", linewidth=2, label=f"Max Limit ({SECTOR_MAX_WEIGHT*100:.0f}%)")
ax.axhline(y=SECTOR_MIN_WEIGHT*100, color="green", linestyle="--", linewidth=2, label=f"Min Limit ({SECTOR_MIN_WEIGHT*100:.0f}%)")

ax.set_xticks(x)
ax.set_xticklabels(sectors, rotation=45, ha="right")
ax.set_ylabel("Weight (%)", fontsize=12)
ax.set_title("Sector Weights Comparison (4 Optimizers)", fontsize=14, fontweight="bold")
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.show()


## 10. แสดงน้ำหนักสินทรัพย์แต่ละตัว


In [229]:
# สร้างตาราง Weights เปรียบเทียบสินทรัพย์รายตัว
weights_df = pd.DataFrame({
    "Ticker": stock_list,
    "Sector": [stock_sector_map[s] for s in stock_list],
    "Unconstrained (%)": (w_unconstrained * 100).round(2),
    "SLSQP (%)": (w_constrained_old * 100).round(2),
    "PSO (%)": (w_pso * 100).round(2),
    "DE (%)": (w_de * 100).round(2),
})

print("=" * 80)
print("📊 Individual Asset Weights (Top 10 in DE Portfolio)")
print("=" * 80)
top10 = weights_df.sort_values(by="DE (%)", ascending=False).head(10)
print(top10.to_string(index=False))


📊 Individual Asset Weights (Top 10 in DE Portfolio)
  Ticker                 Sector  Unconstrained (%)  SLSQP (%)  PSO (%)  DE (%)
  KTB.BK            Thai Stocks              21.88      15.00    12.05   11.69
    CBOE             Financials              12.41      15.00    12.05   11.65
     EME            Industrials               7.65       7.58     5.89    9.75
     CAH            Health Care               7.29      10.12     6.33    8.74
     HWM            Industrials               5.57       7.24    11.52    6.00
  TTB.BK            Thai Stocks               5.65       8.00    10.49    5.86
     MCK            Health Care               2.52       3.98    10.03    4.95
    CIEN Information Technology               2.62       2.46     1.94    4.28
TISCO.BK            Thai Stocks              21.02       1.00     1.52    4.22
 TCAP.BK            Thai Stocks               0.00       1.00     1.56    4.15


In [230]:
# === Bar Chart: Individual Asset Weights ===

# เรียงตาม Sector เพื่อให้ดูง่าย
weights_df_sorted = weights_df.sort_values(by=["Sector", "Ticker"])

fig, ax = plt.subplots(figsize=(18, 8))
x = np.arange(len(weights_df_sorted))
bw = 0.2

ax.bar(x - 1.5*bw, weights_df_sorted["Unconstrained (%)"], bw, label="Unconstrained", color="#9E9E9E", alpha=0.7)
ax.bar(x - 0.5*bw, weights_df_sorted["SLSQP (%)"], bw, label="SLSQP", color="#2196F3", alpha=0.9)
ax.bar(x + 0.5*bw, weights_df_sorted["PSO (%)"], bw, label="PSO", color="#FF9800", alpha=0.9)
ax.bar(x + 1.5*bw, weights_df_sorted["DE (%)"], bw, label="DE (Winner)", color="#4CAF50", alpha=0.9)

ax.axhline(y=STOCK_MAX_WEIGHT*100, color="red", linestyle="--", alpha=0.5, label=f"Max Limit ({STOCK_MAX_WEIGHT*100:.0f}%)")
ax.axhline(y=STOCK_MIN_WEIGHT*100, color="green", linestyle="--", alpha=0.5, label=f"Min Limit ({STOCK_MIN_WEIGHT*100:.0f}%)")

ax.set_xticks(x)
ax.set_xticklabels(weights_df_sorted["Ticker"], rotation=90, fontsize=9)
ax.set_ylabel("Weight (%)", fontsize=12)
ax.set_title("Individual Asset Weights Comparison", fontsize=14, fontweight="bold")
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis="y")

# เพิ่มสีพื้นหลังแบ่ง Sector
current_sector = ""
start_idx = 0
colors = plt.cm.Pastel1(np.linspace(0, 1, 10))
color_idx = 0

for i, row in enumerate(weights_df_sorted.itertuples()):
    if row.Sector != current_sector:
        if i > 0:
            ax.axvspan(start_idx - 0.5, i - 0.5, color=colors[color_idx % 10], alpha=0.2)
            ax.text((start_idx + i - 1) / 2, ax.get_ylim()[1] * 0.95, current_sector, 
                    ha="center", va="top", rotation=90, fontsize=8, alpha=0.7)
            color_idx += 1
        current_sector = row.Sector
        start_idx = i

# ล็อตสุดท้าย
ax.axvspan(start_idx - 0.5, len(weights_df_sorted) - 0.5, color=colors[color_idx % 10], alpha=0.2)
ax.text((start_idx + len(weights_df_sorted) - 1) / 2, ax.get_ylim()[1] * 0.95, current_sector, 
        ha="center", va="top", rotation=90, fontsize=8, alpha=0.7)

plt.tight_layout()
plt.show()


## 11. Efficient Frontier (เปรียบเทียบ)


In [231]:
def generate_efficient_frontier(mean_returns, cov_matrix, sector_constraint, n_points=30):
    """สร้าง Efficient Frontier"""
    n = len(mean_returns)
    
    # หา Min/Max Return ที่เป็นไปได้
    target_returns = np.linspace(
        mean_returns.min() * 252,
        mean_returns.max() * 252,
        n_points
    )
    
    frontier_vol = []
    frontier_ret = []
    
    if sector_constraint:
        bounds = tuple((STOCK_MIN_WEIGHT, STOCK_MAX_WEIGHT) for _ in range(n))
    else:
        bounds = tuple((0, 1) for _ in range(n))
    
    for target in target_returns:
        constraints = get_base_constraints()
        constraints.append({
            'type': 'eq',
            'fun': lambda w, t=target: np.sum(w * mean_returns) * 252 - t
        })
        if sector_constraint:
            constraints += get_sector_constraints(
                sector_stock_indices, SECTOR_MAX_WEIGHT, SECTOR_MIN_WEIGHT
            )
        
        init = np.array(n * [1./n])
        try:
            result = minimize(
                minimize_volatility, init,
                args=(mean_returns, cov_matrix),
                method='SLSQP',
                bounds=bounds,
                constraints=constraints,
                options={'maxiter': 1000}
            )
            if result.success:
                r, v, _ = portfolio_performance(result.x, mean_returns, cov_matrix)
                frontier_vol.append(v)
                frontier_ret.append(r)
        except:
            continue
    
    return frontier_vol, frontier_ret


print("⏳ Generating Efficient Frontiers...")
ef_vol_uc, ef_ret_uc = generate_efficient_frontier(mean_returns, cov_matrix, False)
ef_vol_c, ef_ret_c = generate_efficient_frontier(mean_returns, cov_matrix, True)
print("✅ Done!")


⏳ Generating Efficient Frontiers...
✅ Done!


In [232]:
# === Plot Efficient Frontier with 3 Optimizers ===

fig, ax = plt.subplots(figsize=(16, 9))

# Plot Unconstrained Frontier
ax.plot([v*100 for v in ef_vol_uc], [r*100 for r in ef_ret_uc],
        "gray", linestyle="--", linewidth=2, label="Unconstrained Frontier")

# Plot Unconstrained Max Sharpe
ax.scatter(vol_uc*100, ret_uc*100, marker="*", color="black", s=300, zorder=5, 
           label=f"Unconstrained Obj (SR={sharpe_uc:.3f})")

# Plot the 3 constrained optimizers
ax.scatter(slsqp_vol*100, slsqp_ret*100, marker="o", color="#2196F3", s=200, zorder=6, 
           label=f"SLSQP (SR={slsqp_sharpe:.3f})")
ax.scatter(pso_vol*100, pso_ret*100, marker="D", color="#FF9800", s=200, zorder=6, 
           label=f"PSO Hybrid (SR={pso_sharpe:.3f})")
ax.scatter(de_vol*100, de_ret*100, marker="*", color="#4CAF50", s=400, zorder=7, 
           label=f"DE Hybrid (SR={de_sharpe:.3f})")

# Individual assets
for i, t in enumerate(stock_list):
    ax.scatter(np.sqrt(cov_annual.iloc[i,i])*100, mu_annual.iloc[i]*100,
               s=30, c="gray", alpha=0.4)
    ax.annotate(t, (np.sqrt(cov_annual.iloc[i,i])*100, mu_annual.iloc[i]*100),
                fontsize=7, alpha=0.6)

ax.set_title("Efficient Frontier & Optimizer Positions", fontsize=15, fontweight="bold")
ax.set_xlabel("Annual Volatility (%)", fontsize=12)
ax.set_ylabel("Annual Return (%)", fontsize=12)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 12. Backtest: เปรียบเทียบ Portfolio Performance


In [233]:
# คำนวณ portfolio returns
port_returns_uc = (returns * w_unconstrained).sum(axis=1)
port_returns_slsqp = (returns * w_constrained_old).sum(axis=1)
port_returns_pso = (returns * w_pso).sum(axis=1)
port_returns_de = (returns * w_de).sum(axis=1)

# Equal-weight benchmark
w_equal = np.array([1/n_stocks] * n_stocks)
port_returns_eq = (returns * w_equal).sum(axis=1)


In [234]:
# === สรุปผล Backtest (5 Portfolios) ===

def portfolio_metrics(daily_returns, name):
    total_return = (1 + daily_returns).prod() - 1
    annual_ret = daily_returns.mean() * 252
    annual_vol = daily_returns.std() * np.sqrt(252)
    sharpe = (annual_ret - RISK_FREE_RATE) / annual_vol
    
    # Sortino Ratio
    downside_returns = daily_returns[daily_returns < 0]
    downside_vol = downside_returns.std() * np.sqrt(252)
    sortino = (annual_ret - RISK_FREE_RATE) / downside_vol if downside_vol > 0 else 0
    
    cum_returns = (1 + daily_returns).cumprod()
    peak = cum_returns.cummax()
    drawdown = (cum_returns - peak) / peak
    max_dd = drawdown.min()
    calmar = annual_ret / abs(max_dd) if max_dd != 0 else 0
    
    return [name, total_return, annual_ret, annual_vol, sharpe, sortino, max_dd, calmar]

results = [
    portfolio_metrics(port_returns_uc, "Unconstrained"),
    portfolio_metrics(port_returns_slsqp, "SLSQP Constrained"),
    portfolio_metrics(port_returns_pso, "PSO Constrained"),
    portfolio_metrics(port_returns_de, "DE Constrained"),
    portfolio_metrics(port_returns_eq, "Equal Weight")
]

columns = ["Portfolio", "Total Return (%)", "Annual Return (%)", "Annual Vol (%)", "Sharpe Ratio", "Sortino Ratio", "Max Drawdown (%)", "Calmar Ratio"]
results_df = pd.DataFrame(results, columns=columns)

# Format as percentage
for col in ["Total Return (%)", "Annual Return (%)", "Annual Vol (%)", "Max Drawdown (%)"]:
    results_df[col] = (results_df[col] * 100).round(2)

results_df["Sharpe Ratio"] = results_df["Sharpe Ratio"].round(3)
results_df["Sortino Ratio"] = results_df["Sortino Ratio"].round(3)
results_df["Calmar Ratio"] = results_df["Calmar Ratio"].round(3)

print("=" * 105)
print("📊 สรุปผล Backtest Performance (All Optimizers)")
print("=" * 105)
print(results_df.to_string(index=False))

# Visualize Cumulative Returns
plt.figure(figsize=(16, 7))
plt.plot((1 + port_returns_uc).cumprod() * 100, label="Unconstrained", color="#9E9E9E", alpha=0.7)
plt.plot((1 + port_returns_slsqp).cumprod() * 100, label="SLSQP Constrained", color="#2196F3", linewidth=2)
plt.plot((1 + port_returns_pso).cumprod() * 100, label="PSO Constrained", color="#FF9800", linewidth=2)
plt.plot((1 + port_returns_de).cumprod() * 100, label="DE Constrained", color="#4CAF50", linewidth=3)
plt.plot((1 + port_returns_eq).cumprod() * 100, label="Equal Weight", color="#9C27B0", linestyle="--")

plt.title("Cumulative Returns (Base = 100 THB)", fontsize=14, fontweight="bold")
plt.ylabel("Value (THB)", fontsize=12)
plt.xlabel("Date", fontsize=12)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


📊 สรุปผล Backtest Performance (All Optimizers)
        Portfolio  Total Return (%)  Annual Return (%)  Annual Vol (%)  Sharpe Ratio  Sortino Ratio  Max Drawdown (%)  Calmar Ra
tio
    Unconstrained            241.60              34.80           12.01         2.523          3.546             -9.00         3.
869
SLSQP Constrained            306.51              40.04           15.05         2.361          3.407            -10.32         3.
880
  PSO Constrained            296.47              39.33           14.97         2.327          3.366             -9.55         4.
118
   DE Constrained            300.52              39.67           15.30         2.298          3.299            -10.67         3.
719
     Equal Weight            327.64              42.06           18.67         2.011          2.852            -15.64         2.
688


## 13. Transaction Costs (TC) — ต้นทุนในการทำธุรกรรม

### ประเภทต้นทุน:
| ประเภท | รายละเอียด | ค่าเริ่มต้น |
|--------|-----------|------------|
| **Broker Fee** | ค่าธรรมเนียมโบรกเกอร์ | 0.10% |
| **Tax** | ภาษีหัก ณ ที่จ่าย | 0.05% |
| **Bid-Ask Spread** | ส่วนต่างราคา (Liquidity Cost) | 0.05% |
| **รวม (Total TC)** | ต้นทุนต่อการซื้อขาย 1 ครั้ง | **0.20%** |


In [235]:
# ==========================================================
# Transaction Costs (TC) Analysis
# ==========================================================

# --- TC Parameters ---
BROKER_FEE = 0.0010    # 0.10% ค่าธรรมเนียมโบรกเกอร์
TAX = 0.0005           # 0.05% ภาษี
BID_ASK_SPREAD = 0.0005  # 0.05% ส่วนต่างราคา
TOTAL_TC = BROKER_FEE + TAX + BID_ASK_SPREAD  # 0.20% per trade
REBALANCE_FREQ = 21    # ปรับพอร์ตทุก 21 Days (รายเดือน)

print("=" * 65)
print("Transaction Costs (TC) Configuration")
print("=" * 65)
print(f"  Broker Fee     : {BROKER_FEE*100:.2f}%")
print(f"  Tax            : {TAX*100:.2f}%")
print(f"  Bid-Ask Spread : {BID_ASK_SPREAD*100:.2f}%")
print(f"  Total TC/trade : {TOTAL_TC*100:.2f}%")
print(f"  Rebalance Freq : Every {REBALANCE_FREQ} trading days")

# --- Backtest with TC ---
def backtest_with_tc(returns, weights, tc_rate, rebal_days):
    """Backtest with transaction costs on rebalancing days"""
    n_days = len(returns)
    port_values = [1.0]  # start with 1.0
    current_weights = weights.copy()
    total_tc_paid = 0
    n_rebalances = 0
    daily_tc = []
    
    for day in range(n_days):
        daily_ret = returns.iloc[day].values
        
        # Portfolio return for the day
        port_ret = np.sum(current_weights * daily_ret)
        
        # Update weights based on returns (drift)
        new_vals = current_weights * (1 + daily_ret)
        current_weights = new_vals / new_vals.sum()
        
        # Rebalance check
        tc_cost = 0
        if (day + 1) % rebal_days == 0:
            # Calculate turnover (how much weights changed)
            turnover = np.sum(np.abs(current_weights - weights))
            tc_cost = turnover * tc_rate
            total_tc_paid += tc_cost
            current_weights = weights.copy()
            n_rebalances += 1
        
        new_value = port_values[-1] * (1 + port_ret - tc_cost)
        port_values.append(new_value)
        daily_tc.append(tc_cost)
    
    return np.array(port_values[1:]), total_tc_paid, n_rebalances, daily_tc

# Run backtests
print("\nRunning backtests...")

# Without TC
cum_no_tc_uc = (1 + (returns * w_unconstrained).sum(axis=1)).cumprod()
cum_no_tc_c = (1 + (returns * w_constrained).sum(axis=1)).cumprod()

# With TC
cum_tc_uc, tc_paid_uc, n_reb_uc, _ = backtest_with_tc(returns, w_unconstrained, TOTAL_TC, REBALANCE_FREQ)
cum_tc_c, tc_paid_c, n_reb_c, _ = backtest_with_tc(returns, w_constrained, TOTAL_TC, REBALANCE_FREQ)

print(f"  Rebalances: {n_reb_uc} times over {len(returns)} days")
print(f"  Total TC paid (Unconstrained): {tc_paid_uc*100:.2f}%")
print(f"  Total TC paid (Constrained):   {tc_paid_c*100:.2f}%")


Transaction Costs (TC) Configuration
  Broker Fee     : 0.10%
  Tax            : 0.05%
  Bid-Ask Spread : 0.05%
  Total TC/trade : 0.20%
  Rebalance Freq : Every 21 trading days
Running backtests...
  Rebalances: 43 times over 909 days
  Total TC paid (Unconstrained): 0.42%
  Total TC paid (Constrained):   0.49%


In [236]:
# === TC Impact: Comparison Table & Charts ===

def calc_metrics(cum_values, label):
    total_ret = cum_values[-1] / cum_values[0] - 1
    days = len(cum_values)
    ann_ret = (1 + total_ret)**(252/days) - 1
    
    daily_rets = pd.Series(cum_values).pct_change().dropna()
    ann_vol = daily_rets.std() * np.sqrt(252)
    sharpe = (ann_ret - RISK_FREE_RATE) / ann_vol if ann_vol > 0 else 0
    
    peak = pd.Series(cum_values).cummax()
    dd = (pd.Series(cum_values) - peak) / peak
    max_dd = dd.min()
    
    return [label, total_ret*100, ann_ret*100, ann_vol*100, sharpe, max_dd*100]

tc_results = []

# 1. Unconstrained
cum_uc_notc, _, _, _ = backtest_with_tc(returns, w_unconstrained, 0.0, REBALANCE_FREQ)
cum_uc_tc, _, _, _ = backtest_with_tc(returns, w_unconstrained, TOTAL_TC, REBALANCE_FREQ)
tc_results.append(calc_metrics(cum_uc_notc, "Unconstrained (NO TC)"))
tc_results.append(calc_metrics(cum_uc_tc, "Unconstrained (WITH TC)"))

# 2. SLSQP
cum_slsqp_notc, _, _, _ = backtest_with_tc(returns, w_constrained_old, 0.0, REBALANCE_FREQ)
cum_slsqp_tc, _, _, _ = backtest_with_tc(returns, w_constrained_old, TOTAL_TC, REBALANCE_FREQ)
tc_results.append(calc_metrics(cum_slsqp_notc, "SLSQP (NO TC)"))
tc_results.append(calc_metrics(cum_slsqp_tc, "SLSQP (WITH TC)"))

# 3. PSO
cum_pso_notc, _, _, _ = backtest_with_tc(returns, w_pso, 0.0, REBALANCE_FREQ)
cum_pso_tc, _, _, _ = backtest_with_tc(returns, w_pso, TOTAL_TC, REBALANCE_FREQ)
tc_results.append(calc_metrics(cum_pso_notc, "PSO (NO TC)"))
tc_results.append(calc_metrics(cum_pso_tc, "PSO (WITH TC)"))

# 4. DE
cum_de_notc, _, _, _ = backtest_with_tc(returns, w_de, 0.0, REBALANCE_FREQ)
cum_de_tc, _, _, _ = backtest_with_tc(returns, w_de, TOTAL_TC, REBALANCE_FREQ)
tc_results.append(calc_metrics(cum_de_notc, "DE (NO TC)"))
tc_results.append(calc_metrics(cum_de_tc, "DE (WITH TC)"))

tc_df = pd.DataFrame(tc_results, columns=["Portfolio", "Total Return (%)", "Annual Return (%)", "Annual Vol (%)", "Sharpe Ratio", "Max DD (%)"])
tc_df = tc_df.round(3)

print("=" * 85)
print("Transaction Costs Impact: Before vs After TC")
print("=" * 85)
print(tc_df.to_string(index=False))

drag_uc = tc_df.iloc[0]["Total Return (%)"] - tc_df.iloc[1]["Total Return (%)"]
drag_slsqp = tc_df.iloc[2]["Total Return (%)"] - tc_df.iloc[3]["Total Return (%)"]
drag_pso = tc_df.iloc[4]["Total Return (%)"] - tc_df.iloc[5]["Total Return (%)"]
drag_de = tc_df.iloc[6]["Total Return (%)"] - tc_df.iloc[7]["Total Return (%)"]

print("\nTC Drag (กำไรที่หายไปจาก TC สำหรับระยะเวลา 4 ปี):")
print(f"  Unconstrained: {drag_uc:.2f}% of total return lost")
print(f"  SLSQP:         {drag_slsqp:.2f}% of total return lost")
print(f"  PSO:           {drag_pso:.2f}% of total return lost")
print(f"  DE:            {drag_de:.2f}% of total return lost")

# Bar Chart comparison
fig, ax = plt.subplots(figsize=(10, 5))
drags = [drag_uc, drag_slsqp, drag_pso, drag_de]
labels = ["Unconstrained", "SLSQP", "PSO", "DE"]
colors = ["#9E9E9E", "#2196F3", "#FF9800", "#4CAF50"]

bars = ax.bar(labels, drags, color=colors, alpha=0.8)
ax.set_ylabel("Total Return Lost to TC (%)")
ax.set_title("Transaction Cost Drag by Optimizer", fontweight="bold")

for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 0.05,
            f"{height:.2f}%", ha="center", va="bottom", fontweight="bold")

plt.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()


Transaction Costs Impact: Before vs After TC
              Portfolio  Total Return (%)  Annual Return (%)  Annual Vol (%)  Sharpe Ratio  Max DD (%)
  Unconstrained (NO TC)           234.538             39.763          12.035         2.930      -9.165
Unconstrained (WITH TC)           233.132             39.599          12.035         2.917      -9.173
          SLSQP (NO TC)           303.595             47.226          15.035         2.842     -10.432
        SLSQP (WITH TC)           301.638             47.028          15.035         2.829     -10.452
            PSO (NO TC)           294.018             46.249          14.950         2.793      -9.731
          PSO (WITH TC)           292.134             46.055          14.950         2.780      -9.739
             DE (NO TC)           300.248             46.887          15.296         2.771     -10.739
           DE (WITH TC)           298.290             46.687          15.296         2.758     -10.762
TC Drag (กำไรที่หายไปจาก TC 

## 14. Transaction Lots (TL) — หน่วยการซื้อขาย

สินทรัพย์บางอย่างไม่สามารถซื้อเป็นทศนิยมได้ ต้องซื้อเป็น **จำนวนเต็ม** หรือ **Lot**

| ตลาด | Lot Size | ตัวอย่าง |
|------|----------|--------|
| 🇹🇭 SET (หุ้นไทย) | **100 หุ้น/Lot** | ซื้อ DELTA.BK ต้องซื้อ 100, 200, 300... |
| 🇺🇸 US Stocks | **1 หุ้น** | ซื้อ AAPL ได้ 1, 2, 3... หุ้น |
| 🇺🇸 US ETFs | **1 หน่วย** | ซื้อ GLD ได้ 1, 2, 3... หน่วย |
| ₿ Crypto | **ทศนิยมได้** | ซื้อ BTC 0.001 ได้ |


In [237]:
print("=" * 80)
print(f"Transaction Lots Analysis (Portfolio = {PORTFOLIO_VALUE:,.0f} THB)")
print("=" * 80)

latest_prices = data.iloc[-1]
lot_sizes = lot_size_map.copy()

def adjust_to_lots(weights, prices, port_value, lot_sizes):
    results = []
    total_invested = 0
    
    for i, ticker in enumerate(prices.index):
        target_value = weights[i] * port_value
        price = prices[ticker]
        lot = lot_sizes.get(ticker, 1)
        
        if lot < 1:
            shares = round(target_value / price, 4)
        else:
            raw_shares = target_value / price
            shares = int(raw_shares / lot) * lot
        
        actual_value = shares * price
        total_invested += actual_value
        
        results.append({
            "Ticker": ticker,
            "Sector": stock_sector_map[ticker],
            "Lot Size": lot,
            "Price": round(price, 2),
            "Target Wt (%)": round(weights[i] * 100, 2),
            "Target Value": round(target_value, 0),
            "Shares": shares,
            "Actual Value": round(actual_value, 0),
            "Actual Wt (%)": 0,
        })
    
    for r in results:
        r["Actual Wt (%)"] = round(r["Actual Value"] / total_invested * 100, 2)
    
    return pd.DataFrame(results), total_invested

lots_df, total_inv = adjust_to_lots(w_constrained, latest_prices, PORTFOLIO_VALUE, lot_sizes)

print(f"Sector-Constrained (DE) Portfolio (Lot-Adjusted):")
print("-" * 80)
print(lots_df.to_string(index=False))
print(f"Total Invested: {total_inv:>15,.0f} THB")
print(f"Cash Remaining: {PORTFOLIO_VALUE - total_inv:>15,.0f} THB")
print(f"Cash %:         {(PORTFOLIO_VALUE - total_inv)/PORTFOLIO_VALUE*100:>14.2f}%")


Transaction Lots Analysis (Portfolio = 1,000,000 THB)
Sector-Constrained (DE) Portfolio (Lot-Adjusted):
--------------------------------------------------------------------------------
  Ticker                 Sector  Lot Size    Price  Target Wt (%)  Target Value  Shares  Actual Value  Actual Wt (%)
     CAH            Health Care       1.0  6918.18          10.12      101245.0    14.0       96854.0          11.08
    CBOE             Financials       1.0  9328.65          15.00      150000.0    16.0      149258.0          17.08
     CEG              Utilities       1.0 10380.22           2.00       20000.0     1.0       10380.0           1.19
    CIEN Information Technology       1.0 13532.62           2.46       24641.0     1.0       13533.0           1.55
     COR            Health Care       1.0 10881.07           1.00       10000.0     0.0           0.0           0.00
     EME            Industrials       1.0 24643.62           7.58       75764.0     3.0       73931.0           8

In [238]:
# === Lot Adjustment Impact: ความแตกต่าง Target vs Actual ===

# Weight deviation
lots_df["Wt Diff (%)"] = lots_df["Actual Wt (%)"] - lots_df["Target Wt (%)"]

tracking_error = np.sqrt(np.sum(lots_df["Wt Diff (%)"].values**2))
max_dev = lots_df["Wt Diff (%)"].abs().max()
avg_dev = lots_df["Wt Diff (%)"].abs().mean()

print("=" * 60)
print("Lot Adjustment Impact")
print("=" * 60)
print(f"  Tracking Error (RMSE):    {tracking_error:.3f}%")
print(f"  Max Weight Deviation:     {max_dev:.3f}%")
print(f"  Avg Weight Deviation:     {avg_dev:.3f}%")

# Chart
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 1) Target vs Actual weights
x = np.arange(len(lots_df))
w_bar = 0.35
axes[0].barh(x - w_bar/2, lots_df["Target Wt (%)"], w_bar, label="Target (Optimal)", color="#3498db", alpha=0.8)
axes[0].barh(x + w_bar/2, lots_df["Actual Wt (%)"], w_bar, label="Actual (Lot-Adjusted)", color="#e74c3c", alpha=0.8)
axes[0].set_yticks(x)
axes[0].set_yticklabels(lots_df["Ticker"], fontsize=9)
axes[0].set_xlabel("Weight (%)")
axes[0].set_title("Target vs Actual Weights (Lot-Adjusted)", fontsize=12, fontweight="bold")
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3, axis="x")

# 2) Weight deviation
colors_dev = ["#27ae60" if v >= 0 else "#e74c3c" for v in lots_df["Wt Diff (%)"]]
axes[1].barh(lots_df["Ticker"], lots_df["Wt Diff (%)"], color=colors_dev, alpha=0.8)
axes[1].axvline(x=0, color="black", linewidth=1)
axes[1].set_xlabel("Weight Deviation (%)")
axes[1].set_title(f"Weight Deviation from Optimal\n(RMSE={tracking_error:.3f}%)", fontsize=12, fontweight="bold")
axes[1].grid(True, alpha=0.3, axis="x")

plt.tight_layout()
plt.show()

print("\n>> สรุป: การปรับ Lot ทำให้น้ำหนักเบี่ยงเบนจาก Optimal เล็กน้อย")
print(f"   แต่ยังอยู่ในเกณฑ์ที่ยอมรับได้ (RMSE = {tracking_error:.3f}%)")


Lot Adjustment Impact
  Tracking Error (RMSE):    4.360%
  Max Weight Deviation:     2.080%
  Avg Weight Deviation:     0.772%


>> สรุป: การปรับ Lot ทำให้น้ำหนักเบี่ยงเบนจาก Optimal เล็กน้อย
   แต่ยังอยู่ในเกณฑ์ที่ยอมรับได้ (RMSE = 4.360%)


## 15. Short Sell with Leverage Constraint

### แนวคิด:
- **Long-Only**: ซื้ออย่างเดียว (w ≥ 0), ∑w = 1
- **Long-Short**: ซื้อ (Long) + ขายชอร์ต (Short) ได้, ∑w = 1 แต่ ∑|w| > 1
- **Leverage**: Gross Exposure = ∑|w| → ถ้า > 1 แปลว่าขยายสถานะเกินทุนจริง

### ตัวอย่าง:
| สถานะ | Long | Short | Net (∑w) | Gross (∑\|w\|) | Leverage |
|--------|------|-------|----------|---------------|----------|
| Long-Only | 100% | 0% | 100% | 100% | 1.0x |
| Long-Short (free) | 120% | -20% | 100% | 140% | 1.4x |
| Long-Short (limited) | 110% | -10% | 100% | **≤ 130%** | ≤ 1.3x |


In [239]:
# ==========================================================
# Short Sell with Leverage Constraint
# ==========================================================

from scipy.optimize import minimize as sp_min

mu = returns.mean() * 252
cov_mat = returns.cov() * 252
n = len(mu)

# --- Leverage Levels to Compare ---
LEVERAGE_LEVELS = {
    "Long-Only (1.0x)":        {"bounds": (0, 1),       "max_gross": 1.0},
    "Long-Short (1.3x)":       {"bounds": (-0.15, 0.3), "max_gross": 1.3},
    "Long-Short (1.6x)":       {"bounds": (-0.20, 0.4), "max_gross": 1.6},
    "Long-Short (2.0x)":       {"bounds": (-0.30, 0.5), "max_gross": 2.0},
    "Long-Short (No Limit)":   {"bounds": (-0.50, 1.0), "max_gross": 99},
}

def optimize_with_leverage(mu, cov_mat, bounds_range, max_gross, n_trials=30):
    """Optimize Max Sharpe with leverage constraint"""
    n = len(mu)
    bounds = tuple((bounds_range[0], bounds_range[1]) for _ in range(n))
    
    constraints = [
        {"type": "eq", "fun": lambda w: np.sum(w) - 1},  # net weight = 1
    ]
    if max_gross < 50:  # add leverage constraint
        constraints.append({
            "type": "ineq",
            "fun": lambda w: max_gross - np.sum(np.abs(w))  # gross <= max
        })
    
    def neg_sharpe(w):
        ret = np.dot(w, mu)
        vol = np.sqrt(np.dot(w.T, np.dot(cov_mat, w)))
        return -(ret - RISK_FREE_RATE) / vol if vol > 0 else 0
    
    best = None
    best_val = np.inf
    for _ in range(n_trials):
        init = rng.dirichlet(np.ones(n))
        try:
            res = sp_min(neg_sharpe, init, method="SLSQP",
                        bounds=bounds, constraints=constraints,
                        options={"maxiter": 1000, "ftol": 1e-12})
            if res.success and res.fun < best_val:
                best_val = res.fun
                best = res
        except:
            continue
    return best

# --- Run All Leverage Levels ---
print("=" * 80)
print("Short Sell with Leverage Constraint: Comparison")
print("=" * 80)
print()

results_lev = []
weights_lev = {}

for name, params in LEVERAGE_LEVELS.items():
    print(f"Optimizing: {name}...")
    res = optimize_with_leverage(mu, cov_mat, params["bounds"], params["max_gross"])
    w = res.x
    
    ret = np.dot(w, mu)
    vol = np.sqrt(np.dot(w.T, np.dot(cov_mat, w)))
    sharpe = (ret - RISK_FREE_RATE) / vol
    gross = np.sum(np.abs(w))
    long_sum = np.sum(w[w > 0])
    short_sum = np.sum(w[w < 0])
    n_short = np.sum(w < -0.005)
    
    results_lev.append({
        "Strategy": name,
        "Return (%)": round(ret * 100, 2),
        "Vol (%)": round(vol * 100, 2),
        "Sharpe": round(sharpe, 3),
        "Long (%)": round(long_sum * 100, 1),
        "Short (%)": round(short_sum * 100, 1),
        "Gross (%)": round(gross * 100, 1),
        "N Short": int(n_short),
    })
    weights_lev[name] = w

lev_df = pd.DataFrame(results_lev)
print()
print(lev_df.to_string(index=False))


Short Sell with Leverage Constraint: Comparison
Optimizing: Long-Only (1.0x)...
Optimizing: Long-Short (1.3x)...
Optimizing: Long-Short (1.6x)...
Optimizing: Long-Short (2.0x)...
Optimizing: Long-Short (No Limit)...
             Strategy  Return (%)  Vol (%)  Sharpe  Long (%)  Short (%)  Gross (%)  N Short
     Long-Only (1.0x)       34.96    12.09   2.520     100.0        0.0      100.0        0
    Long-Short (1.3x)       35.65    12.07   2.581     115.0      -15.0      130.0        4
    Long-Short (1.6x)       37.12    12.57   2.595     129.4      -29.4      158.8        5
    Long-Short (2.0x)       37.12    12.57   2.595     129.4      -29.4      158.8        5
Long-Short (No Limit)       37.12    12.57   2.595     129.4      -29.4      158.8        5


In [240]:
# === Short Sell & Leverage: Visualization ===

fig, axes = plt.subplots(2, 2, figsize=(18, 12))

# 1) Sharpe vs Leverage
axes[0,0].plot(lev_df["Gross (%)"], lev_df["Sharpe"], "bo-", linewidth=2, markersize=10)
for _, row in lev_df.iterrows():
    axes[0,0].annotate(row["Strategy"].split("(")[1].rstrip(")"),
                      (row["Gross (%)"], row["Sharpe"]),
                      fontsize=9, ha="left", va="bottom")
axes[0,0].set_xlabel("Gross Exposure (%)", fontsize=12)
axes[0,0].set_ylabel("Sharpe Ratio", fontsize=12)
axes[0,0].set_title("Sharpe Ratio vs Leverage Level", fontsize=13, fontweight="bold")
axes[0,0].grid(True, alpha=0.3)

# 2) Return vs Vol scatter
colors_lev = plt.cm.viridis(np.linspace(0, 0.9, len(results_lev)))
for j, (name, row) in enumerate(zip(lev_df["Strategy"], results_lev)):
    axes[0,1].scatter(row["Vol (%)"], row["Return (%)"], s=200,
                     c=[colors_lev[j]], edgecolors="black", linewidth=1.5, zorder=5)
    axes[0,1].annotate(name.split("(")[1].rstrip(")"),
                      (row["Vol (%)"], row["Return (%)"]),
                      fontsize=9, ha="left", va="bottom")
axes[0,1].set_xlabel("Volatility (%)", fontsize=12)
axes[0,1].set_ylabel("Return (%)", fontsize=12)
axes[0,1].set_title("Risk-Return: Different Leverage Levels", fontsize=13, fontweight="bold")
axes[0,1].grid(True, alpha=0.3)

# 3) Weights comparison: Long-Only vs Long-Short 1.3x
if "Long-Only (1.0x)" in weights_lev and "Long-Short (1.3x)" in weights_lev:
    w_lo = weights_lev["Long-Only (1.0x)"]
    w_ls = weights_lev["Long-Short (1.3x)"]
    x_idx = np.arange(n)
    bw = 0.35
    axes[1,0].bar(x_idx - bw/2, w_lo * 100, bw, label="Long-Only", color="#3498db", alpha=0.8)
    axes[1,0].bar(x_idx + bw/2, w_ls * 100, bw, label="Long-Short (1.3x)", color="#e74c3c", alpha=0.8)
    axes[1,0].axhline(y=0, color="black", linewidth=1)
    axes[1,0].set_xticks(x_idx)
    axes[1,0].set_xticklabels(mu.index, rotation=45, ha="right", fontsize=8)
    axes[1,0].set_ylabel("Weight (%)")
    axes[1,0].set_title("Long-Only vs Long-Short Weights", fontsize=13, fontweight="bold")
    axes[1,0].legend(fontsize=10)
    axes[1,0].grid(True, alpha=0.3, axis="y")

# 4) Stacked bar: Long vs Short composition
strategies = [r["Strategy"].split("(")[1].rstrip(")") for r in results_lev]
longs = [r["Long (%)"] for r in results_lev]
shorts = [-r["Short (%)"] for r in results_lev]  # make positive for display
x_s = np.arange(len(strategies))
axes[1,1].bar(x_s, longs, 0.6, label="Long (%)", color="#27ae60", alpha=0.8)
axes[1,1].bar(x_s, shorts, 0.6, bottom=longs, label="Short (%)", color="#e74c3c", alpha=0.8)
axes[1,1].set_xticks(x_s)
axes[1,1].set_xticklabels(strategies, fontsize=10)
axes[1,1].set_ylabel("Exposure (%)")
axes[1,1].set_title("Gross Exposure Breakdown", fontsize=13, fontweight="bold")
axes[1,1].legend(fontsize=10)
for i, (l, s) in enumerate(zip(longs, shorts)):
    axes[1,1].text(i, l + s + 2, f"{l+s:.0f}%", ha="center", fontsize=10, fontweight="bold")
axes[1,1].grid(True, alpha=0.3, axis="y")

plt.suptitle("Short Sell with Leverage Constraint Analysis",
             fontsize=16, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

print("\nสรุป:")
print("  - Leverage สูงขึ้น -> Sharpe อาจดีขึ้น แต่ Gross Exposure สูง (เสี่ยงมาก)")
print("  - Leverage 1.3x เป็น sweet spot: ได้ประโยชน์จาก short sell แต่ไม่เสี่ยงเกินไป")
print("  - No Limit -> optimizer ใช้ leverage มากเกินไป ไม่เหมาะใช้จริง")


สรุป:
  - Leverage สูงขึ้น -> Sharpe อาจดีขึ้น แต่ Gross Exposure สูง (เสี่ยงมาก)
  - Leverage 1.3x เป็น sweet spot: ได้ประโยชน์จาก short sell แต่ไม่เสี่ยงเกินไป
  - No Limit -> optimizer ใช้ leverage มากเกินไป ไม่เหมาะใช้จริง


## Out-of-Sample Testing (Train/Test Split)

Split data into **Train (3 years)** and **Test (1 year)** to check for overfitting:
- Optimize weights on training data
- Test performance on unseen data
- Compare In-Sample vs Out-of-Sample results


In [241]:
# ==========================================================
# Out-of-Sample Testing: Train (3yr) / Test (1yr)
# ==========================================================

split_idx = int(len(returns) * 0.75)  # 75% train, 25% test
train_returns = returns.iloc[:split_idx]
test_returns = returns.iloc[split_idx:]

print("=" * 70)
print("Out-of-Sample Testing: Train/Test Split")
print("=" * 70)
print(f"  Train period: {train_returns.index[0].strftime('%Y-%m-%d')} to {train_returns.index[-1].strftime('%Y-%m-%d')} ({len(train_returns)} days)")
print(f"  Test period:  {test_returns.index[0].strftime('%Y-%m-%d')} to {test_returns.index[-1].strftime('%Y-%m-%d')} ({len(test_returns)} days)")

# --- Optimize on TRAIN data ---
train_mu = train_returns.mean() * 252
train_cov = train_returns.cov() * 252

# Define Optimization Functions for Train set
def slsqp_train():
    def neg_sharpe(w):
        ret = np.dot(w, train_mu)
        vol = np.sqrt(np.dot(w.T, np.dot(train_cov, w)))
        return -((ret - RISK_FREE_RATE) / vol) if vol > 0 else 0
    
    n = len(train_mu)
    bounds = tuple((STOCK_MIN_WEIGHT, STOCK_MAX_WEIGHT) for _ in range(n))
    cons = [{"type": "eq", "fun": lambda w: np.sum(w) - 1}]
    for sector in set(stock_sector_map.values()):
        idx = [i for i, t in enumerate(stock_list) if stock_sector_map[t] == sector]
        cons.append({"type": "ineq", "fun": lambda w, idx=idx: np.sum(w[idx]) - SECTOR_MIN_WEIGHT})
        cons.append({"type": "ineq", "fun": lambda w, idx=idx: SECTOR_MAX_WEIGHT - np.sum(w[idx])})
        
    best_w = None
    best_val = np.inf
    for _ in range(15):
        init = rng.dirichlet(np.ones(n))
        try:
            res = sp_minimize(neg_sharpe, init, method="SLSQP", bounds=bounds, constraints=cons)
            if res.success and res.fun < best_val:
                best_val = res.fun
                best_w = res.x
        except:
            pass
    return best_w, -best_val

def pso_train():
    def objective(w):
        w_norm = w / np.sum(w) if np.sum(w) > 0 else np.ones(n_assets)/n_assets
        ret = np.dot(w_norm, train_mu)
        vol = np.sqrt(np.dot(w_norm.T, np.dot(train_cov, w_norm)))
        if vol < 1e-10: return 999
        sharpe = (ret - RISK_FREE_RATE) / vol
        
        penalty = 0
        for sector in set(stock_sector_map.values()):
            idx = [i for i, t in enumerate(stock_list) if stock_sector_map[t] == sector]
            sw = np.sum(w_norm[idx])
            if sw > SECTOR_MAX_WEIGHT: penalty += 100*(sw - SECTOR_MAX_WEIGHT)**2
            if sw < SECTOR_MIN_WEIGHT: penalty += 100*(SECTOR_MIN_WEIGHT - sw)**2
        for wi in w_norm:
            if wi > STOCK_MAX_WEIGHT: penalty += 50*(wi - STOCK_MAX_WEIGHT)**2
            if wi < STOCK_MIN_WEIGHT: penalty += 50*(STOCK_MIN_WEIGHT - wi)**2
        return -sharpe + penalty

    lb = np.ones(n_assets) * STOCK_MIN_WEIGHT
    ub = np.ones(n_assets) * STOCK_MAX_WEIGHT
    
    w_raw, fopt = pso(objective, lb, ub, swarmsize=100, maxiter=200, debug=False)
    w_norm = w_raw / np.sum(w_raw)
    
    # Calculate raw Sharpe
    ret = np.dot(w_norm, train_mu)
    vol = np.sqrt(np.dot(w_norm.T, np.dot(train_cov, w_norm)))
    sharpe = (ret - RISK_FREE_RATE) / vol if vol > 0 else 0
    return w_norm, sharpe

def de_train():
    def objective(w):
        w_norm = w / np.sum(w) if np.sum(w) > 0 else np.ones(n_assets)/n_assets
        ret = np.dot(w_norm, train_mu)
        vol = np.sqrt(np.dot(w_norm.T, np.dot(train_cov, w_norm)))
        if vol < 1e-10: return 999
        sharpe = (ret - RISK_FREE_RATE) / vol
        
        penalty = 0
        for sector in set(stock_sector_map.values()):
            idx = [i for i, t in enumerate(stock_list) if stock_sector_map[t] == sector]
            sw = np.sum(w_norm[idx])
            if sw > SECTOR_MAX_WEIGHT: penalty += 100*(sw - SECTOR_MAX_WEIGHT)**2
            if sw < SECTOR_MIN_WEIGHT: penalty += 100*(SECTOR_MIN_WEIGHT - sw)**2
        for wi in w_norm:
            if wi > STOCK_MAX_WEIGHT: penalty += 50*(wi - STOCK_MAX_WEIGHT)**2
            if wi < STOCK_MIN_WEIGHT: penalty += 50*(STOCK_MIN_WEIGHT - wi)**2
        return -sharpe + penalty
        
    bounds_de = [(STOCK_MIN_WEIGHT, STOCK_MAX_WEIGHT) for _ in range(n_assets)]
    res_de = differential_evolution(objective, bounds_de, maxiter=50, popsize=10, disp=False, polish=False)
    
    w_raw = res_de.x
    w_norm = w_raw / np.sum(w_raw)
    
    # Calculate raw Shape
    ret = np.dot(w_norm, train_mu)
    vol = np.sqrt(np.dot(w_norm.T, np.dot(train_cov, w_norm)))
    sharpe = (ret - RISK_FREE_RATE) / vol if vol > 0 else 0
    return w_norm, sharpe

print("\nRunning Optimizers on TRAIN data (3 years)...")
t0 = time.time()
w_slsqp_train, sr_slsqp_train = slsqp_train()
time_slsqp = time.time() - t0
print(f"  SLSQP Train Sharpe: {sr_slsqp_train:.4f}  |  Time: {time_slsqp:.2f}s")

t0 = time.time()
w_pso_train, sr_pso_train = pso_train()
time_pso = time.time() - t0
print(f"  PSO   Train Sharpe: {sr_pso_train:.4f}  |  Time: {time_pso:.2f}s")

t0 = time.time()
w_de_train, sr_de_train = de_train()
time_de = time.time() - t0
print(f"  DE    Train Sharpe: {sr_de_train:.4f}  |  Time: {time_de:.2f}s")

# --- Evaluate on TRAIN vs TEST ---
def eval_portfolio(w, ret_data, label, optimizer):
    port_ret = (ret_data * w).sum(axis=1)
    cum = (1 + port_ret).cumprod()
    ann_ret = port_ret.mean() * 252
    ann_vol = port_ret.std() * np.sqrt(252)
    sharpe = (ann_ret - RISK_FREE_RATE) / ann_vol
    peak = cum.cummax()
    dd = (cum - peak) / peak
    max_dd = dd.min()
    return {
        "Optimizer": optimizer,
        "Period": label,
        "Ann Return": round(ann_ret * 100, 2),
        "Ann Vol": round(ann_vol * 100, 2),
        "Sharpe": round(sharpe, 3),
        "Max DD": round(max_dd * 100, 2),
        "Cum Return": round((cum.iloc[-1] - 1) * 100, 2),
    }

oos_results = []
# SLSQP
oos_results.append(eval_portfolio(w_slsqp_train, train_returns, "Train", "SLSQP"))
oos_results.append(eval_portfolio(w_slsqp_train, test_returns, "Test", "SLSQP"))
# PSO
oos_results.append(eval_portfolio(w_pso_train, train_returns, "Train", "PSO"))
oos_results.append(eval_portfolio(w_pso_train, test_returns, "Test", "PSO"))
# DE
oos_results.append(eval_portfolio(w_de_train, train_returns, "Train", "DE"))
oos_results.append(eval_portfolio(w_de_train, test_returns, "Test", "DE"))

oos_df = pd.DataFrame(oos_results)
print("\n" + "=" * 90)
print("In-Sample (Train) vs Out-of-Sample (Test) Performance by Optimizer")
print("=" * 90)
print(oos_df.to_string(index=False))

# Overfitting Check (Sharpe Degradation)
print("\nSharpe Degradation (Train -> Test):")
for opt in ["SLSQP", "PSO", "DE"]:
    opt_data = oos_df[oos_df["Optimizer"] == opt]
    train_sr = opt_data[opt_data["Period"] == "Train"]["Sharpe"].values[0]
    test_sr = opt_data[opt_data["Period"] == "Test"]["Sharpe"].values[0]
    degradation = (train_sr - test_sr) / train_sr * 100 if train_sr > 0 else 0
    status = "GOOD" if degradation < 30 else ("MODERATE" if degradation < 50 else "OVERFITTING")
    print(f"  {opt:5s}: {degradation:6.1f}%  -> {status}")


Out-of-Sample Testing: Train/Test Split
  Train period: 2022-04-27 to 2025-03-25 (681 days)
  Test period:  2025-03-26 to 2026-03-19 (228 days)
Running Optimizers on TRAIN data (3 years)...
  SLSQP Train Sharpe: 2.1727  |  Time: 0.73s
Stopping search: Swarm best objective change less than 1e-08
  PSO   Train Sharpe: 2.1386  |  Time: 1.16s
  DE    Train Sharpe: 2.1076  |  Time: 1.30s
In-Sample (Train) vs Out-of-Sample (Test) Performance by Optimizer
Optimizer Period  Ann Return  Ann Vol  Sharpe  Max DD  Cum Return
    SLSQP  Train       35.74    14.38   2.173   -7.21      155.32
    SLSQP   Test       49.44    16.75   2.683   -9.81       54.38
      PSO  Train       35.28    14.39   2.139   -7.58      152.16
      PSO   Test       45.63    16.88   2.436   -9.01       49.13
       DE  Train       34.68    14.32   2.108   -7.25      148.16
       DE   Test       48.34    16.89   2.595   -9.36       52.82
Sharpe Degradation (Train -> Test):
  SLSQP:  -23.5%  -> GOOD
  PSO  :  -13.9%  -> GO

In [242]:
# === Out-of-Sample: Visualization (3 Optimizers) ===

fig, axes = plt.subplots(1, 2, figsize=(20, 7))

# 1) Cumulative returns: Train vs Test
train_cum_slsqp = (1 + (train_returns * w_slsqp_train).sum(axis=1)).cumprod()
test_cum_slsqp = (1 + (test_returns * w_slsqp_train).sum(axis=1)).cumprod()
test_cum_slsqp_norm = test_cum_slsqp / test_cum_slsqp.iloc[0] * train_cum_slsqp.iloc[-1]

train_cum_pso = (1 + (train_returns * w_pso_train).sum(axis=1)).cumprod()
test_cum_pso = (1 + (test_returns * w_pso_train).sum(axis=1)).cumprod()
test_cum_pso_norm = test_cum_pso / test_cum_pso.iloc[0] * train_cum_pso.iloc[-1]

train_cum_de = (1 + (train_returns * w_de_train).sum(axis=1)).cumprod()
test_cum_de = (1 + (test_returns * w_de_train).sum(axis=1)).cumprod()
test_cum_de_norm = test_cum_de / test_cum_de.iloc[0] * train_cum_de.iloc[-1]

axes[0].plot(train_cum_slsqp.index, train_cum_slsqp.values, color="#2196F3", linestyle="-", alpha=0.9, label="SLSQP (Train)")
axes[0].plot(test_cum_slsqp_norm.index, test_cum_slsqp_norm.values, color="#2196F3", linestyle="--", alpha=0.9, label="SLSQP (Test)")

axes[0].plot(train_cum_pso.index, train_cum_pso.values, color="#FF9800", linestyle="-", alpha=0.9, label="PSO (Train)")
axes[0].plot(test_cum_pso_norm.index, test_cum_pso_norm.values, color="#FF9800", linestyle="--", alpha=0.9, label="PSO (Test)")

axes[0].plot(train_cum_de.index, train_cum_de.values, color="#4CAF50", linestyle="-", linewidth=2.5, alpha=0.9, label="DE (Train)")
axes[0].plot(test_cum_de_norm.index, test_cum_de_norm.values, color="#4CAF50", linestyle="--", linewidth=2.5, alpha=0.9, label="DE (Test)")

axes[0].axvline(x=train_cum_slsqp.index[-1], color="black", linestyle="-.", linewidth=2, alpha=0.7, label="TRAIN / TEST SPLIT")

axes[0].set_title("Out-of-Sample Test: Cumulative Returns (Base=100)", fontsize=13, fontweight="bold")
axes[0].set_ylabel("Cumulative Return")
axes[0].legend(fontsize=9, loc="upper left")
axes[0].grid(True, alpha=0.3)

# 2) Bar chart: Sharpe degradation
degradations = []
for opt in ["SLSQP", "PSO", "DE"]:
    opt_data = oos_df[oos_df["Optimizer"] == opt]
    train_sr = opt_data[opt_data["Period"] == "Train"]["Sharpe"].values[0]
    test_sr = opt_data[opt_data["Period"] == "Test"]["Sharpe"].values[0]
    deg = (train_sr - test_sr) / train_sr * 100 if train_sr > 0 else 0
    degradations.append(deg)

x = np.arange(3)
colors = ["#2196F3", "#FF9800", "#4CAF50"]
bars = axes[1].bar(x, degradations, color=colors, alpha=0.8)
axes[1].axhline(y=0, color="black", linewidth=1)
axes[1].axhline(y=30, color="red", linestyle="--", alpha=0.5, label="Warning Threshold (30%)")

axes[1].set_xticks(x)
axes[1].set_xticklabels(["SLSQP", "PSO", "DE"])
axes[1].set_title("Sharpe Degradation (Train to Test)\nLower is better (Negative means Test > Train)", fontsize=13, fontweight="bold")
axes[1].set_ylabel("Degradation (%)")
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis="y")

for i, bar in enumerate(bars):
    height = bar.get_height()
    y_pos = height + 1 if height >= 0 else height - 4
    axes[1].text(bar.get_x() + bar.get_width()/2., y_pos, f"{degradations[i]:.1f}%", 
                 ha="center", va="bottom", fontweight="bold")

plt.suptitle("Model Evaluation: Generalization on Unseen Data (1 Year Test Set)", fontsize=16, fontweight="bold", y=1.05)
plt.tight_layout()
plt.show()
